# PTMCMC_Neighbour_Optimisation

This notebook develops, validates, and benchmarks a neighbour-restricted optimisation for the Parallel Tempering Monte Carlo (PTMCMC) code used with RuNNer neural-network potentials.

The original motivation was to reduce the computational cost of symmetry-function updates performed after a trial atom move. In the existing implementation, radial symmetry updates scale approximately as O(N) while angular symmetry updates scale approximately as O(N²), even though the RuNNer cutoff function causes many of the resulting contributions to evaluate to zero. The optimisation developed here exploits the existing cutoff radius to identify only those atoms whose symmetry values can change following a move, allowing the update to be restricted to a local neighbourhood around the moved atom.

The notebook began with the development and validation of a standalone neighbour-search algorithm using small test clusters. This was subsequently extended to:

* benchmark neighbour-search performance across a range of cluster sizes,
* analyse neighbourhood statistics for realistic copper clusters,
* generate and validate Mackay icosahedral copper clusters containing up to several hundred atoms,
* compare neighbour counts and neighbourhood fractions as cluster size increases,
* implement neighbour-restricted radial and angular symmetry updates,
* verify that the neighbour-restricted algorithm reproduces the symmetry matrices produced by the original PTMCMC implementation,
* integrate the optimisation into the PTMCMC energy-update pathway,
* benchmark the neighbour-restricted and original implementations for correctness and computational performance.

A central goal of the notebook is to determine how the computational cost of PTMCMC simulations scales with cluster size when symmetry-function updates are restricted to atoms within the cutoff neighbourhood of a moved atom. This optimisation is intended to enable simulations of substantially larger clusters than are currently computationally feasible while preserving numerical agreement with the existing implementation.


In [7]:
using Pkg
Pkg.activate("/home/rev/ParallelTemperingMonteCarlo.jl")
Pkg.instantiate()

using LinearAlgebra
using StaticArrays
using Pkg
using Random
using BenchmarkTools


n_atoms = 13

pos_ne13 = [
    [2.825384495892464, 0.928562467914040, 0.505520149314310],
    [2.023342172678102, -2.136126268595355, 0.666071287554958],
    [2.033761811732818, -0.643989413759464, -2.133000349161121],
    [0.979777205108572, 2.312002562803556, -1.671909307631893],
    [0.962914279874254, -0.102326586625353, 2.857083360096907],
    [0.317957619634043, 2.646768968413408, 1.412132053672896],
    [-2.825388342924982, -0.928563755928189, -0.505520471387560],
    [-0.317955944853142, -2.646769840660271, -1.412131825293682],
    [-0.979776174195320, -2.312003751825495, 1.671909138648006],
    [-0.962916072888105, 0.102326392265998, -2.857083272537599],
    [-2.023340541398004, 2.136128558801072, -0.666071089291685],
    [-2.033762834001679, 0.643989905095452, 2.132999911364582],
    [0.000002325340981, 0.000000762100600, 0.000000414930733]
]

AtoBohr = 1.8897259886
pos_ne13 = pos_ne13 .* AtoBohr
positions = [SVector{3,Float64}(p) for p in pos_ne13]
#=
function find_neighbours(positions, atomindex, cutoff)
    neighbours = Int[]
    distances = Float64[]
    cutoff2 = cutoff^2
    xi = positions[atomindex]

    for j in eachindex(positions)
        if j != atomindex
            d2 = sum((xi .- positions[j]).^2)
            if d2 <= cutoff2
                push!(neighbours, j)
                push!(distances, sqrt(d2))
            end
        end
    end

    return neighbours, distances
end
=#

function find_neighbours!(
    neighbours::Vector{Int},
    distances::Vector{Float64},
    positions,
    atomindex::Int,
    cutoff::Float64
)
    cutoff2 = cutoff^2
    xi = positions[atomindex]
    k = 0

    for j in eachindex(positions)
        if j != atomindex
            d = xi - positions[j]
            d2 = d[1]^2 + d[2]^2 + d[3]^2

            if d2 <= cutoff2
                k += 1
                neighbours[k] = j
                distances[k] = sqrt(d2)
            end
        end
    end

    return k
end

  Activating project at `~/ParallelTemperingMonteCarlo.jl`


find_neighbours! (generic function with 1 method)

In [14]:
for cutoff in [0.1, 1.0, 3.0, 5.0, 6.0, 7.0, 10.0]
    N = length(positions)

    neighbours = Vector{Int}(undef, N - 1)
    distances = Vector{Float64}(undef, N - 1)

    k = find_neighbours!(neighbours, distances, positions, 5, cutoff)
    
    println("cutoff = ", cutoff)
    println("neighbours = ", neighbours[1:k])
    println("distances = ", distances[1:k])
    println("count = ", k)
    println()
end

cutoff = 0.1
neighbours = Int64[]
distances = Float64[]
count = 0

cutoff = 1.0
neighbours = Int64[]
distances = Float64[]
count = 0

cutoff = 3.0
neighbours = Int64[]
distances = Float64[]
count = 0

cutoff = 5.0
neighbours = Int64[]
distances = Float64[]
count = 0

cutoff = 6.0
neighbours = [1, 2, 6, 9, 12, 13]
distances = [5.994149199782839, 5.994150773064821, 5.994149992030653, 5.994143309903633, 5.9941437750255115, 5.700772216012668]
count = 6

cutoff = 7.0
neighbours = [1, 2, 6, 9, 12, 13]
distances = [5.994149199782839, 5.994150773064821, 5.994149992030653, 5.994143309903633, 5.9941437750255115, 5.700772216012668]
count = 6

cutoff = 10.0
neighbours = [1, 2, 3, 4, 6, 7, 8, 9, 11, 12, 13]
distances = [5.994149199782839, 5.994150773064821, 9.6987410609835, 9.698740270126057, 5.994149992030653, 9.69873382019968, 9.698735505705573, 5.994143309903633, 9.69873518631835, 5.9941437750255115, 5.700772216012668]
count = 11



In [ ]:
function reference_neighbours(positions, i, cutoff)
    result = Int[]
    for j in eachindex(positions)
        if j != i
            r = norm(positions[j] - positions[i])
            if r <= cutoff
                push!(result, j)
            end
        end
    end
    return sort(result)
end

function test_neighbour_function(positions, cutoff)
    N = length(positions)

    neighbours = Vector{Int}(undef, N - 1)
    distances = Vector{Float64}(undef, N - 1)

    for i in eachindex(positions)
        k = find_neighbours!(neighbours, distances, positions, i, cutoff)

        n1 = sort(neighbours[1:k])
        n2 = reference_neighbours(positions, i, cutoff)

        if n1 != n2
            println("Mismatch for atom ", i)
            println("find_neighbours!: ", n1)
            println("reference:        ", n2)
            return false
        end
    end

    return true
end

#=function reference_neighbours(positions, i, cutoff)
    result = Int[]
    for j in eachindex(positions)
        if j != i
            r = norm(positions[j] - positions[i])
            if r <= cutoff
                push!(result, j)
            end
        end
    end
    return sort(result)
end

function test_neighbour_function(positions, cutoff)
    for i in eachindex(positions)
        n1, _ = find_neighbours(positions, i, cutoff)
        n2 = reference_neighbours(positions, i, cutoff)

        if sort(n1) != n2
            println("Mismatch for atom ", i)
            println("find_neighbours: ", sort(n1))
            println("reference:       ", n2)
            return false
        end
    end
    return true
end
=#

test_neighbour_function (generic function with 1 method)

In [16]:
test_neighbour_function(positions, 6.0)

true

In [17]:
function write_xyz(filename, positions; element="Ne")
    open(filename, "w") do io
        println(io, length(positions))
        println(io, "Ne13 cluster")
        for p in positions
            println(io, "$element $(p[1]) $(p[2]) $(p[3])")
        end
    end
end

write_xyz (generic function with 1 method)

testing allocations. Test error bars for run times over man calculations, plot ft vs N. 

## Export for VESTA visualisation

The following function writes the Ne13 positions to an `.xyz` file. This allows the cluster to be opened in VESTA so the geometry can be inspected visually. The visualisation is only a sanity check; the mathematical correctness test above is the main validation of the neighbour-search function.

In [18]:
write_xyz("ne13.xyz", positions)

Create random clusters of different sizes:

In [10]:
#=
function random_cluster(N; box_radius=10.0) 
    return [SVector{3,Float64}(box_radius .* (2rand(3) .- 1)) for _ in 1:N] 
end
=#

function random_cluster_constant_density(N; base_N=1000, base_box_radius=30.0)
    box_radius = base_box_radius * (N / base_N)^(1/3)

    return [
        SVector{3,Float64}(box_radius .* (2rand(3) .- 1))
        for _ in 1:N
    ]
end

function find_central_atom(positions)
    centre = sum(positions) / length(positions)

    best_index = 1
    best_dist2 = Inf

    for i in eachindex(positions)
        d = positions[i] - centre
        d2 = d[1]^2 + d[2]^2 + d[3]^2

        if d2 < best_dist2
            best_dist2 = d2
            best_index = i
        end
    end

    return best_index
end

find_central_atom (generic function with 1 method)

Then benchmark:

In [11]:

#=
for N in [13, 55, 147, 309, 561, 1000]
    random_positions = random_cluster(N)
    cutoff = 6.0
    atomindex = 1

    neighbours = Vector{Int}(undef, N - 1)
    distances = Vector{Float64}(undef, N - 1)

    println("N = ", N)
    @btime find_neighbours!($neighbours, $distances, $random_positions, $atomindex, $cutoff)
    println()
end
=#

using BenchmarkTools
using Random
using Statistics

Random.seed!(1234)

#Ns = [1000, 2000, 5000, 10000, 20000]
Ns = [13, 55, 147, 309, 561, 1000, 2000, 5000, 10000, 20000]
n_repeats = 10

clusters = Dict{Int, Any}()
neigh_buffers = Dict{Int, Vector{Int}}()
dist_buffers = Dict{Int, Vector{Float64}}()

for N in Ns
    #clusters[N] = random_cluster(N)
    clusters[N] = random_cluster_constant_density(N)
    neigh_buffers[N] = Vector{Int}(undef, N - 1)
    dist_buffers[N] = Vector{Float64}(undef, N - 1)
end

times_ns = Float64[]
times_per_atom = Float64[]

cutoff = 11.338
atomindex = 1

#realistic cutoff from real code, 
#realistc cluster not randomly generated.

for N in Ns
    positions_N = clusters[N]
    neighbours_N = neigh_buffers[N]
    distances_N = dist_buffers[N]
    atomindex = find_central_atom(positions_N)

    repeat_times = Float64[]
    repeat_allocs = Int[]
    repeat_memory = Int[]

    k = find_neighbours!(neighbours_N, distances_N, positions_N, atomindex, cutoff)

    for r in 1:n_repeats
        trial = @benchmark find_neighbours!(
            $neighbours_N,
            $distances_N,
            $positions_N,
            $atomindex,
            $cutoff
        )

        push!(repeat_times, minimum(trial).time)
        push!(repeat_allocs, trial.allocs)
        push!(repeat_memory, trial.memory)
    end

    time_ns = median(repeat_times)
    allocations = maximum(repeat_allocs)
    memory_bytes = maximum(repeat_memory)

    push!(times_ns, time_ns)
    push!(times_per_atom, time_ns / N)

    println("N = ", N)
    println("median of repeated minimum times = ", time_ns, " ns")
    println("neighbour count k = ", k)
    println("neighbour fraction k/(N-1) = ", k / (N - 1))
    println("time per atom = ", time_ns / N, " ns")
    println("allocations: ", allocations)
    println("memory: ", memory_bytes)
    println()
end

println()
println("Successive scaling:")

for i in 2:length(Ns)
    size_ratio = Ns[i] / Ns[i-1]
    runtime_ratio = times_ns[i] / times_ns[i-1]
    per_atom_ratio = times_per_atom[i] / times_per_atom[i-1]
    empirical_power = log(runtime_ratio) / log(size_ratio)

    println("N: ", Ns[i-1], " → ", Ns[i])
    println("size ratio: N[i]/N[i-1] = ", Ns[i], "/", Ns[i-1], " = ", size_ratio)
    println("runtime ratio: RT[i]/RT[i-1] = ", times_ns[i], "/", times_ns[i-1], " = ", runtime_ratio)
    println("per atom ratio: (RT[i]/N[i]) / (RT[i-1]/N[i-1]) = ", per_atom_ratio)
    println("empirical exponent p ≈ ", empirical_power)
    println()
end


#=
using BenchmarkTools
using Random

Random.seed!(1234)

#Ns = [13, 55, 147, 309, 561, 1000]

Ns = [1000, 2000, 5000, 10000, 20000]

times_ns = Float64[]
times_per_atom = Float64[]

for N in Ns
    random_positions = random_cluster(N)
    cutoff = 6.0
    atomindex = 1

    neighbours = Vector{Int}(undef, N - 1)
    distances = Vector{Float64}(undef, N - 1)

    result = @benchmark find_neighbours!(
        $neighbours,
        $distances,
        $random_positions,
        $atomindex,
        $cutoff
    )

    median_time_ns = median(result).time
    push!(times_ns, median_time_ns)
    time_per_atom = median_time_ns/N
    push!(times_per_atom, time_per_atom)
    memory_bytes = result.memory
    allocations = result.allocs

    println("N = ", N)
    println("median time = ", median_time_ns, " ns")
    println("time per atom ns: ", time_per_atom)
    println("allocations: ", allocations)
    println("memory: ", memory_bytes)
    println()
end

# Compute the ratios of of successive runtime per atom's for increasing cluster sizes.


println()
println("Successive scaling:")

for i in 2:length(Ns)
    
    println()
    size_ratio = Ns[i] / Ns[i-1]
    per_atom_ratio = times_per_atom[i] / times_per_atom[i-1]

    println("N: ", Ns[i-1], " → ", Ns[i])
    println("size ratio: N[i]/N[i-1] = ", Ns[i], "/", Ns[i-1],  " = ", Ns[i]/Ns[i-1])
    println("per atom ratio = ", per_atom_ratio)

     println("N: ", Ns[i-1], " → ", Ns[i])
    println("size ratio = ", size_ratio)
    println("runtime ratio = ", runtime_ratio)
    println("per atom ratio = ", per_atom_ratio)
    println("empirical exponent p ≈ ", empirical_power)
    println()
end
=#

N = 13
median of repeated minimum times = 47.48737373737374 ns
neighbour count k = 12
neighbour fraction k/(N-1) = 1.0
time per atom = 3.652874902874903 ns
allocations: 0
memory: 0

N = 55
median of repeated minimum times = 151.46220382036773 ns
neighbour count k = 29
neighbour fraction k/(N-1) = 0.5370370370370371
time per atom = 2.7538582512794134 ns
allocations: 0
memory: 0

N = 147
median of repeated minimum times = 322.45338983050846 ns
neighbour count k = 29
neighbour fraction k/(N-1) = 0.19863013698630136
time per atom = 2.1935604750374726 ns
allocations: 0
memory: 0

N = 309
median of repeated minimum times = 630.0 ns
neighbour count k = 34
neighbour fraction k/(N-1) = 0.11038961038961038
time per atom = 2.0388349514563107 ns
allocations: 0
memory: 0

N = 561
median of repeated minimum times = 1110.0 ns
neighbour count k = 35
neighbour fraction k/(N-1) = 0.0625
time per atom = 1.9786096256684491 ns
allocations: 0
memory: 0

N = 1000
median of repeated minimum times = 2077.77777

Add a simple artificial move:

In [5]:
function move_atom(positions, atomindex, displacement)
    new_positions = copy(positions)
    new_positions[atomindex] = positions[atomindex] + displacement
    return new_positions
end

move_atom (generic function with 1 method)

Test it:

In [ ]:
cutoff = 6.0
atomindex = 4
displacement = SVector{3,Float64}(0.1, 0.0, 0.0)

old_positions = positions
new_positions = move_atom(positions, atomindex, displacement)

N = length(positions)

old_neigh_buffer = Vector{Int}(undef, N - 1)
old_dist_buffer = Vector{Float64}(undef, N - 1)

new_neigh_buffer = Vector{Int}(undef, N - 1)
new_dist_buffer = Vector{Float64}(undef, N - 1)

k_old = find_neighbours!(old_neigh_buffer, old_dist_buffer, old_positions, atomindex, cutoff)
k_new = find_neighbours!(new_neigh_buffer, new_dist_buffer, new_positions, atomindex, cutoff)

old_neigh = old_neigh_buffer[1:k_old]
new_neigh = new_neigh_buffer[1:k_new]

entered = setdiff(new_neigh, old_neigh)
left = setdiff(old_neigh, new_neigh)
stayed = intersect(old_neigh, new_neigh)

println("old = ", old_neigh)
println("new = ", new_neigh)
println("entered = ", entered)
println("left = ", left)
println("stayed = ", stayed)

#=
cutoff = 6.0
atomindex = 4
displacement = SVector{3,Float64}(0.1, 0.0, 0.0)

old_positions = positions
new_positions = move_atom(positions, atomindex, displacement)

old_neigh, _ = find_neighbours(old_positions, atomindex, cutoff)
new_neigh, _ = find_neighbours(new_positions, atomindex, cutoff)

entered = setdiff(new_neigh, old_neigh)
left = setdiff(old_neigh, new_neigh)
stayed = intersect(old_neigh, new_neigh)

println("old = ", old_neigh)
println("new = ", new_neigh)
println("entered = ", entered)
println("left = ", left)
println("stayed = ", stayed)
=#

old = [1, 3, 6, 10, 11, 13]
new = [1, 3, 13]
entered = Int64[]
left = [6, 10, 11]
stayed = [1, 3, 13]


In [ ]:
write_xyz("ne13_moved.xyz", new_positions)

In [12]:
using Pkg

Pkg.activate("/home/rev/ParallelTemperingMonteCarlo.jl")

using ParallelTemperingMonteCarlo
using DelimitedFiles

X = [ 11              0.001   0.000  11.338
 10              0.001   0.000  11.338
 11              0.020   0.000  11.338
 10              0.020   0.000  11.338
 11              0.035   0.000  11.338
 10              0.035   0.000  11.338
 11              0.100   0.000  11.338
 10              0.100   0.000  11.338
 11              0.400   0.000  11.338
 10              0.400   0.000  11.338]

radsymmvec = RadialType2{Float64}[]
angularsymmvec = AngularType3{Float64}[]

#--------------------------------------------#
#--------Vector of angular symm values-------#
#--------------------------------------------#
V = [[0.0001,1,1,11.338],[0.0001,-1,2,11.338],[0.003,-1,1,11.338],[0.003,-1,2,11.338],[0.008,-1,1,11.338],[0.008,-1,2,11.338],[0.008,1,2,11.338],[0.015,1,1,11.338],[0.015,-1,2,11.338],[0.015,-1,4,11.338],[0.015,-1,16,11.338],[0.025,-1,1,11.338],[0.025,1,1,11.338],[0.025,1,2,11.338],[0.025,-1,4,11.338],[0.025,-1,16,11.338],[0.025,1,16,11.338],[0.045,1,1,11.338],[0.045,-1,2,11.338],[0.045,-1,4,11.338],[0.045,1,4,11.338],[0.045,1,16,11.338],[0.08,1,1,11.338],[0.08,-1,2,11.338],[0.08,-1,4,11.338],[0.08,1,4,11.338]]

T = [111,110,100]

angularsymmvec = AngularType3{Float64}[]
#-------------------------------------------#
#-----------Including scaling data----------#
#-------------------------------------------#
data_path = "/home/rev/ParallelTemperingMonteCarlo.jl/scripts/data"
file = open(joinpath(data_path,"scaling.data")) # full path "./data/scaling.data"
scalingvalues = readdlm(file)
close(file)
G_value_vec = []

for row in eachrow(scalingvalues[1:88,:])
    max_min = [row[4],row[3]]
    push!(G_value_vec,max_min)
end


for symmindex in eachindex(eachrow(X))
    row = X[symmindex,:]
    radsymm = RadialType2{Float64}(row[2],row[4],Int(row[1]),G_value_vec[symmindex])
    push!(radsymmvec,radsymm)
end


let n_index = 10

for element in V
    for types in T

        n_index += 1

        symmfunc = AngularType3{Float64}(element[1],element[2],element[3],11.338,types,G_value_vec[n_index])

        push!(angularsymmvec,symmfunc)
    end
end
end
#---------------------------------------------------#
#------concatenating radial and angular values------#
#---------------------------------------------------#

totalsymmvec = vcat(radsymmvec,angularsymmvec)

#--------------------------------------------------#
#-----------Initialising the nnp weights-----------#
#--------------------------------------------------#
num_nodes::Vector{Int32} = [88, 20, 20, 1]
activation_functions::Vector{Int32} = [1, 2, 2, 1]
file = open(joinpath(data_path, "weights.029.data"), "r+") # "./data/weights.029.data"
weights=readdlm(file)
close(file)
weights = vec(weights)
nnp = NeuralNetworkPotential(num_nodes,activation_functions,weights)

println(typeof(radsymmvec))
println(typeof(angularsymmvec))
println(eltype(radsymmvec))
println(eltype(angularsymmvec))

pot = RuNNerPotential(nnp,radsymmvec,angularsymmvec)

dist2_mat = find_distance2_mat(positions)
f_matrix = cutoff_function.(sqrt.(dist2_mat), Ref(pot.r_cut))

nrad = length(pot.radsymfunctions)
nang = length(pot.angsymfunctions)

G_full = total_symm_calc(
    positions,
    dist2_mat,
    f_matrix,
    pot.radsymfunctions,
    pot.angsymfunctions,
    nrad,
    nang
)

size(G_full)

G_full[1, :]


  Activating project at `~/ParallelTemperingMonteCarlo.jl`


LoadError: cannot set type for global Main.num_nodes. It already has a value or is already set to a different type.

next goal: maximum(abs.(G_full .- G_neigh))

G_full = total_symm_calc(...)

g_local_for_atom5 = compute_only_atom_i_symmetry_values(...)

G_full[:, 5] ≈ g_local_for_atom5

can use smaller radius just for testing, use cu55 to ensure runtime reasonable. 

n(n-1)/2

In [6]:
ico_55 = [[0.0000006584,       -0.0000019175,        0.0000000505],
[-0.0000005810,       -0.0000004871,        0.6678432175],
[0.1845874248,       -0.5681026047,        0.2986701538],
[-0.4832557457,       -0.3511072166,        0.2986684497],
[-0.4832557570,        0.3511046452,        0.2986669456],
[0.1845874064,        0.5681000550,        0.2986677202],
[0.5973371920,       -0.0000012681,        0.2986697030],
[-0.1845860897,       -0.5681038901,       -0.2986676192],
[-0.5973358752,       -0.0000025669,       -0.2986696020],
[-0.1845861081,        0.5680987696,       -0.2986700528],
[0.4832570624,        0.3511033815,       -0.2986683486],
[0.4832570738,       -0.3511084803,       -0.2986668445],
[0.0000018978,       -0.0000033480,       -0.6678431165],
[-0.0000017969,        0.0000009162,        1.3230014650],
[0.1871182835,       -0.5758942175,        0.9797717078],
[-0.4898861924,       -0.3559221410,       0.9797699802],
[-0.4898862039,        0.3559224872,        0.9797684555],
[0.1871182648,        0.5758945856,        0.9797692407],
[0.6055300485,        0.0000001908,        0.9797712507],
[0.7926501864,       -0.5758950093,        0.6055339635],
[0.3656681761,       -1.1254128670,        0.5916673591],
[-0.3027660545,       -0.9318173412,        0.6055326929],
[-0.9573332453,       -0.6955436707,        0.5916639831],
[-0.9797705418,       -0.0000006364,        0.6055294407],
[-0.9573332679,        0.6955423392,        0.5916610035],
[-0.3027660847,        0.9318160902,        0.6055287012],
[0.3656681396,        1.1254115783,        0.5916625380],
[0.7926501677,        0.5758937939,        0.6055314964],
[1.1833279992,       -0.0000006311,        0.5916664660],
[0.6770051458,       -0.9318186223,        0.0000033028],
[0.0000006771,       -1.1517907207,        0.0000025175],
[-0.6770037988,       -0.9318186442,        0.0000007900],
[-1.0954155825,       -0.3559242494,       -0.0000012200],
[-1.0954155940,        0.3559203788,       -0.0000027447],
[-0.6770038290,        0.9318147872,       -0.0000032017],
[0.0000006397,        1.1517868856,       -0.0000024165],
[0.6770051155,        0.9318148091,       -0.0000006889],
[1.0954168993,        0.3559204143,        0.0000013211],
[1.0954169108,       -0.3559242139,        0.0000028458],
[0.3027674014,       -0.9318199253,       -0.6055286002],
[-0.3656668229,       -1.1254154134,       -0.5916624370],
[-0.7926488510,       -0.5758976290,       -0.6055313954],
[-1.1833266824,       -0.0000032040,       -0.5916663649],
[-0.7926488697,        0.5758911742,       -0.6055338624],
[-0.3656668594,        1.1254090319,       -0.5916672580],
[0.3027673712,        0.9318135061,       -0.6055325919],
[0.9573345621,        0.6955398357,       -0.5916638820],
[0.9797718586,       -0.0000031986,       -0.6055293396],
[0.9573345846,       -0.6955461743,       -0.5916609025],
[-0.1871169480,       -0.5758984207,       -0.9797691397],
[-0.6055287318,       -0.0000040259,       -0.9797711497],
[-0.1871169667,        0.5758903824,       -0.9797716067],
[0.4898875091,        0.3559183059,       -0.9797698792],
[0.4898875207,       -0.3559263223,       -0.9797683545],
[0.0000031136,       -0.0000047513,       -1.3230013639]]

#convert to Bohr
nmtobohr = 18.8973

copperconstant = 0.36258*nmtobohr

pos_cu55 = copperconstant*ico_55

positions_cu55 = [SVector{3,Float64}(p) for p in pos_cu55]

atomindex = find_central_atom(positions_cu55)

N = length(positions_cu55)

neighbours = Vector{Int}(undef, N - 1)
distances = Vector{Float64}(undef, N - 1)

cutoff = 11.338

n_repeats = 10

repeat_times = Float64[]
repeat_allocs = Int[]
repeat_memory = Int[]

for r in 1:n_repeats
    trial = @benchmark find_neighbours!(
        $neighbours,
        $distances,
        $positions_cu55,
        $atomindex,
        $cutoff
    )

    push!(repeat_times, minimum(trial).time)
    push!(repeat_allocs, trial.allocs)
    push!(repeat_memory, trial.memory)
end

cu55_time_ns = median(repeat_times)
allocations = maximum(repeat_allocs)
memory_bytes = maximum(repeat_memory)

k = find_neighbours!(neighbours, distances, positions_cu55, atomindex, cutoff)

println("Cu55 neighbour-search benchmark")
println("N = ", N)
println("central atomindex = ", atomindex)
println("neighbour count k = ", k)
println("median of repeated minimum times = ", cu55_time_ns, " ns")
println("time per atom = ", cu55_time_ns / N, " ns")
println("allocations: ", allocations)
println("memory: ", memory_bytes)

UndefVarError: UndefVarError: `find_central_atom` not defined

In [5]:
pos_cu124 = [[8.70179511, 8.74253542, 15.20569724],
[6.39005732, 9.75358828, 15.25634345],
[6.61765311, 7.32377865, 14.55786662], 
[11.05034427, 7.71147855, 15.07693457], 
[10.68145332, 10.13086906, 14.29873707], 
[8.95215825, 6.28895537, 14.51649622], 
[8.3182353, 11.14240391, 14.36256096], 
[6.88531337, 4.89028036, 13.78055486], 
[6.00700642, 11.23875004, 13.24598256], 
[4.94576416, 8.89361261, 13.36669836], 
[5.15678215, 6.4335647, 12.64394147], 
[13.01190572, 9.06859737, 14.10950973], 
[13.34987945, 6.6843243, 14.87463704], 
[12.63667652, 11.46459509, 13.26536007], 
[11.30412764, 5.28095828, 14.34078155], 
[9.74024858, 8.14863095, 13.02035174], 
[10.28746721, 12.50458077, 13.3917089], 
[7.42492934, 9.16534803, 13.09583556], 
[9.23230078, 3.91385558, 13.70922707], 
[5.99871863, 8.30510419, 11.19266601], 
[7.9454019, 4.23288974, 11.59108509], 
[7.95609414, 12.65935228, 12.31648828], 
[14.5021434, 7.49366285, 12.76369186], 
[14.17290686, 9.90108614, 11.94187368], 
[5.46681702, 4.05279617, 11.8240602], 
[5.66014175, 12.70257168, 11.1593427], 
[4.54210452, 10.38426005, 11.30665058], 
[12.22700298, 13.79132921, 12.37801925], 
[12.0371332, 7.1231784, 12.82870617], 
[3.54205533, 8.01399058, 11.39505893], 
[11.69410876, 9.50025226, 12.05679448], 
[3.80394432, 5.57565363, 10.62929223], 
[9.98809339, 5.7172287, 12.24761517], 
[9.35401233, 10.54281637, 12.18452533], 
[7.66823572, 6.71495781, 12.36714964], 
[7.05224715, 10.64070261, 11.07843206], 
[5.65104755, 9.76388025, 9.11451737], 
[13.46394793, 5.11970212, 12.91077999], 
[6.28366785, 5.86407058, 10.42128804], 
[13.16517302, 7.91527462, 10.68094905], 
[12.06906536, 13.82139289, 9.85390747], 
[11.40006115, 3.70866629, 12.36849139], 
[11.29447089, 11.84245977, 11.18758109], 
[10.74939703, 7.53933543, 10.80139553], 
[9.9442686, 13.96241853, 11.30397983], 
[10.0908909, 4.03334811, 10.217075], 
[8.45920111, 8.56402333, 10.94888414], 
[8.7214727, 6.10793105, 10.17275561], 
[7.70252841, 4.13122729, 9.0286544], 
[12.80691368, 10.26982886, 9.83313148], 
[5.35996431, 14.09420268, 9.06635852], 
[14.62064199, 5.93673309, 10.78051744], 
[13.74219103, 12.25274619, 11.04394633], 
[4.25139147, 11.81985064, 9.18785824], 
[5.19224168, 3.9587829, 9.24325289], 
[3.19591732, 9.47468716, 9.30517704], 
[12.17067381, 5.50180832, 10.8246576], 
[2.22922917, 7.13997595, 9.4133909], 
[9.01357135, 11.99805452, 10.11056794], 
[10.38071115, 9.89334495, 10.00136866], 
[10.84143778, 5.97635916, 8.79536507], 
[7.63011274, 14.06638642, 10.18588487], 
[10.09753096, 7.50694058, 2.54652551], 
[9.85042498, 9.93916444, 3.24723567], 
[7.77292073, 8.53267812, 2.69458888], 
[9.58817491, 12.35418008, 4.09559984], 
[7.50275185, 10.97091934, 3.47579587], 
[5.47181929, 9.55989657, 2.9307034], 
[12.15800674, 8.86684457, 3.16686494], 
[8.09959619, 6.12059934, 3.43624639], 
[11.92626868, 11.28475277, 3.93132957], 
[11.48391584, 6.88215046, 4.53093442], 
[5.78943412, 7.16115928, 3.67221824], 
[9.13990335, 7.98369714, 4.72767755], 
[11.65321274, 13.65804692, 4.76604231], 
[13.63361221, 9.79635252, 5.08176424], 
[8.85227831, 10.43437119, 5.49156314], 
[4.32121899, 8.77076498, 5.04449028], 
[6.81852167, 9.03860191, 4.9315605], 
[6.15189008, 4.77905855, 4.49137352], 
[9.55853473, 13.8755421, 6.1671765], 
[13.35214636, 12.17983553, 5.92073785], 
[7.445587, 12.54877653, 5.50690753], 
[12.96544857, 7.77099474, 6.43394274], 
[5.38568384, 11.10782316, 4.92914914], 
[9.52363242, 5.47983423, 5.4268442], 
[11.18765859, 9.3393078, 5.34072289], 
[4.6389086, 6.33570454, 5.81623564], 
[7.17346737, 6.6240737, 5.69692477], 
[11.89093685, 5.37682781, 6.62961702], 
[10.91035629, 11.75287326, 6.14667574], 
[14.99152964, 10.64142108, 7.12742028], 
[10.48719701, 7.40166499, 6.71666932], 
[5.68840848, 8.2462798, 7.05155978], 
[8.16195962, 8.50681522, 6.91054074], 
[5.08022968, 3.97322628, 6.65498704], 
[14.31214947, 8.63740108, 8.45142959], 
[11.87384464, 13.77701602, 7.28625038], 
[8.81962948, 11.95561778, 7.5519933], 
[12.60024211, 10.23341898, 7.30675897], 
[7.56585175, 4.17321877, 6.4621655], 
[6.74230344, 10.57671906, 6.93573961], 
[3.22701684, 7.95482201, 7.23238437], 
[7.43577079, 14.02161708, 7.6168243], 
[5.34782056, 12.63640266, 6.99800821], 
[6.10350027, 5.82344242, 7.84677631], 
[15.5736398, 8.30138839, 10.58203272], 
[9.91234659, 4.00864968, 7.60500744], 
[11.86741581, 8.28425608, 8.62151276], 
[9.75727653, 13.99129957, 8.73431096], 
[13.59755161, 12.27251861, 8.48127222], 
[8.56222831, 6.07056014, 7.66556326], 
[10.18188844, 9.8498387, 7.5115859], 
[4.22525455, 10.32176994, 7.09526883], 
[7.09992889, 7.71815114, 9.03099606], 
[12.22623493, 3.98160706, 8.80470991], 
[3.61329438, 5.53588185, 8.02960605], 
[13.33864696, 6.28011801, 8.61561831], 
[9.47895188, 7.97375589, 8.83446904], 
[11.12640121, 11.85215529, 8.66431717], 
[15.20944499, 10.64907533, 9.67536612], 
[8.11646901, 10.00845422, 8.92578112], 
[4.67116185, 7.43720734, 9.22307017], 
[6.73827459, 12.04973065, 8.9968049]]

cofm = [
    sum(p[1] for p in pos_cu124),
    sum(p[2] for p in pos_cu124),
    sum(p[3] for p in pos_cu124)
] ./ length(pos_cu124)

for p in pos_cu124
    p .-= cofm
end

positions_cu124 = [SVector{3,Float64}(p) for p in pos_cu124]

atomindex = find_central_atom(positions_cu124)

N = length(positions_cu124)

neighbours = Vector{Int}(undef, N - 1)
distances = Vector{Float64}(undef, N - 1)

cutoff = 11.338

k = find_neighbours!(
    neighbours,
    distances,
    positions_cu124,
    atomindex,
    cutoff
)

println("Cu124 neighbour diagnostic")
println("N = ", N)
println("central atomindex = ", atomindex)
println("neighbour count k = ", k)
println("neighbour fraction k/(N-1) = ", k/(N-1))

Cu124 neighbour diagnostic
N = 124
central atomindex = 119
neighbour count k = 123
neighbour fraction k/(N-1) = 1.0


Explore how alitering both cutoff radius and atom-index influnces the result neigbhourhood size for atoms in the above cu124  Icosahedral cluster. 

In [7]:
for cutoff in [4.0, 6.0, 8.0, 10.0, 11.338, 12.0]

    ks = Int[]

    for atomindex in eachindex(positions_cu124)

        k = find_neighbours!(
            neighbours,
            distances,
            positions_cu124,
            atomindex,
            cutoff
        )

        push!(ks, k)

    end

    println()
    println("cutoff = ", cutoff)
    println("min k = ", minimum(ks))
    println("max k = ", maximum(ks))
    println("mean k = ", mean(ks))
    println("mean fraction = ", mean(ks)/(N-1))

end


cutoff = 4.0
min k = 6
max k = 18
mean k = 12.435483870967742
mean fraction = 0.1011014948859166

cutoff = 6.0
min k = 21
max k = 74
mean k = 40.33870967741935
mean fraction = 0.3279569892473118

cutoff = 8.0
min k = 45
max k = 123
mean k = 73.14516129032258
mean fraction = 0.594676108051403

cutoff = 10.0
min k = 70
max k = 123
mean k = 103.29032258064517
mean fraction = 0.8397587201678469

cutoff = 11.338
min k = 95
max k = 123
mean k = 116.59677419354838
mean fraction = 0.9479412536060844

cutoff = 12.0
min k = 103
max k = 123
mean k = 118.79032258064517
mean fraction = 0.9657749803304485


The below's output demonstrates that cutoff radi above 8.0 (including our PTMCMC cutoff radius of 11.338) include all atoms in the cu124 cluster as neigbhours of the most central atom and so there is no improvement on runtime per atom for the neigbhour search of the central-most atom for these larger radi like there is for radi of 4 and 6.

In [ ]:
using BenchmarkTools
using Statistics

cutoff_values = [4.0, 6.0 , 8.0, 10.0, 11.338, 12.0]
n_repeats = 10

println("Cu124 cutoff scan with neighbour statistics and timing")
println("N = ", N)
println("central atomindex = ", atomindex)
println()

for cutoff in cutoff_values

    ks = Int[]

    for i in eachindex(positions_cu124)
        k_i = find_neighbours!(
            neighbours,
            distances,
            positions_cu124,
            i,
            cutoff
        )

        push!(ks, k_i)
    end

    repeat_times = Float64[]
    repeat_allocs = Int[]
    repeat_memory = Int[]

    for r in 1:n_repeats
        trial = @benchmark find_neighbours!(
            $neighbours,
            $distances,
            $positions_cu124,
            $atomindex,
            $cutoff
        )

        push!(repeat_times, minimum(trial).time)
        push!(repeat_allocs, trial.allocs)
        push!(repeat_memory, trial.memory)
    end

    cutoff_time_ns = median(repeat_times)
    allocations = maximum(repeat_allocs)
    memory_bytes = maximum(repeat_memory)

    central_k = find_neighbours!(
        neighbours,
        distances,
        positions_cu124,
        atomindex,
        cutoff
    )

    println("cutoff = ", cutoff)
    println("central k = ", central_k)
    println("central fraction = ", central_k / (N - 1))
    println("min k = ", minimum(ks))
    println("max k = ", maximum(ks))
    println("mean k = ", mean(ks))
    println("std(k) = ", std(ks))
    println("mean fraction = ", mean(ks) / (N - 1))
    println("median of repeated minimum times = ", cutoff_time_ns, " ns")
    println("time per atom = ", cutoff_time_ns / N, " ns")
    println("allocations: ", allocations)
    println("memory: ", memory_bytes)
    println()
end

Cu124 cutoff scan with neighbour statistics and timing
N = 124
central atomindex = 119

cutoff = 4.0
central k = 12
central fraction = 0.0975609756097561
min k = 6
max k = 18
mean k = 12.435483870967742
std(k) = 3.7918224708106045
mean fraction = 0.1011014948859166
median of repeated minimum times = 184.91207153502233 ns
time per atom = 1.4912263833469543 ns
allocations: 0
memory: 0

cutoff = 6.0
central k = 66
central fraction = 0.5365853658536586
min k = 21
max k = 74
mean k = 40.33870967741935
std(k) = 12.679211009264993
mean fraction = 0.3279569892473118
median of repeated minimum times = 228.76320802818208 ns
time per atom = 1.8448645808724362 ns
allocations: 0
memory: 0

cutoff = 8.0
central k = 123
central fraction = 1.0
min k = 45
max k = 123
mean k = 73.14516129032258
std(k) = 17.814625191271393
mean fraction = 0.594676108051403
median of repeated minimum times = 282.57597173144876 ns
time per atom = 2.278838481705232 ns
allocations: 0
memory: 0

cutoff = 10.0
central k = 123


The below's output however demonstrates that for the production PTMCMC cutoff radius of 11.338 that there is a slight reduced runtime per atom for the mean neigbour search of atoms in the cu124 cluster as not all atoms are neigbhours of all other atoms for this radius. The reduction in runtime per atom is significantly more for smaller radi for which the center-most atom's neigbhour search including all atoms. 

In [6]:
using BenchmarkTools
using Statistics

cutoff_values = [4.0, 6.0, 8.0, 10.0, 11.338, 12.0]
n_repeats = 10

println("Cu124 individual-atom neighbour-search timing by cutoff")
println("N = ", N)
println()

for cutoff in cutoff_values

    ks = Int[]
    atom_times = Float64[]
    atom_allocs = Int[]
    atom_memory = Int[]

    for atomindex in eachindex(positions_cu124)

        k = find_neighbours!(
            neighbours,
            distances,
            positions_cu124,
            atomindex,
            cutoff
        )

        push!(ks, k)

        repeat_times = Float64[]
        repeat_allocs = Int[]
        repeat_memory = Int[]

        for r in 1:n_repeats
            trial = @benchmark find_neighbours!(
                $neighbours,
                $distances,
                $positions_cu124,
                $atomindex,
                $cutoff
            )

            push!(repeat_times, minimum(trial).time)
            push!(repeat_allocs, trial.allocs)
            push!(repeat_memory, trial.memory)
        end

        push!(atom_times, median(repeat_times))
        push!(atom_allocs, maximum(repeat_allocs))
        push!(atom_memory, maximum(repeat_memory))
    end

    println("cutoff = ", cutoff)
    println("min k = ", minimum(ks))
    println("max k = ", maximum(ks))
    println("mean k = ", mean(ks))
    println("std(k) = ", std(ks))
    println("mean fraction = ", mean(ks)/(N-1))
    println("mean single-search time = ", mean(atom_times), " ns")
    println("median single-search time = ", median(atom_times), " ns")
    println("min single-search time = ", minimum(atom_times), " ns")
    println("max single-search time = ", maximum(atom_times), " ns")
    println("mean time per scanned atom = ", mean(atom_times)/N, " ns")
    println("max allocations = ", maximum(atom_allocs))
    println("max memory = ", maximum(atom_memory))
    println()
end

Cu124 individual-atom neighbour-search timing by cutoff
N = 124

cutoff = 4.0
min k = 6
max k = 18
mean k = 12.435483870967742
std(k) = 3.7918224708106045
mean fraction = 0.1011014948859166
mean single-search time = 195.68966467583203 ns
median single-search time = 194.86743786431242 ns
min single-search time = 189.71034902663422 ns
max single-search time = 208.80919003115264 ns
mean time per scanned atom = 1.5781424570631615 ns
max allocations = 0
max memory = 0

cutoff = 6.0
min k = 21
max k = 74
mean k = 40.33870967741935
std(k) = 12.679211009264993
mean fraction = 0.3279569892473118
mean single-search time = 218.49198064114725 ns
median single-search time = 215.71092490842491 ns
min single-search time = 201.52542372881356 ns
max single-search time = 246.1139896373057 ns
mean time per scanned atom = 1.7620321019447358 ns
max allocations = 0
max memory = 0

cutoff = 8.0
min k = 45
max k = 123
mean k = 73.14516129032258
std(k) = 17.814625191271393
mean fraction = 0.594676108051403
mea

In [2]:
using StaticArrays
using LinearAlgebra

function mackay_icosahedron(shell::Int; target_nn_bohr=nothing)
    φ = (1 + sqrt(5.0)) / 2

    verts = SVector{3,Float64}[]

    for s1 in (-1.0, 1.0), s2 in (-1.0, 1.0)
        push!(verts, SVector(0.0, s1, s2*φ))
        push!(verts, SVector(s1, s2*φ, 0.0))
        push!(verts, SVector(s1*φ, 0.0, s2))
    end

    # Find triangular faces of the icosahedron
    edge = minimum(norm(verts[i] - verts[j]) for i in eachindex(verts), j in eachindex(verts) if i < j)

    faces = Tuple{Int,Int,Int}[]

    for i in 1:length(verts)-2
        for j in i+1:length(verts)-1
            for k in j+1:length(verts)
                if abs(norm(verts[i] - verts[j]) - edge) < 1e-8 &&
                   abs(norm(verts[i] - verts[k]) - edge) < 1e-8 &&
                   abs(norm(verts[j] - verts[k]) - edge) < 1e-8
                    push!(faces, (i, j, k))
                end
            end
        end
    end

    # Fill each tetrahedron: centre + one triangular face
    pointset = Set{NTuple{3,Float64}}()

    for (a, b, c) in faces
        v1, v2, v3 = verts[a], verts[b], verts[c]

        for i in 0:shell
            for j in 0:(shell - i)
                for k in 0:(shell - i - j)
                    p = (i*v1 + j*v2 + k*v3) / shell
                    key = (
                        round(p[1], digits=10),
                        round(p[2], digits=10),
                        round(p[3], digits=10)
                    )
                    push!(pointset, key)
                end
            end
        end
    end

    positions = [SVector{3,Float64}(p) for p in pointset]

    # Recentre at origin
    centre = sum(positions) / length(positions)
    positions = [p - centre for p in positions]

    # Optional scaling
    if target_nn_bohr !== nothing
        min_dist = minimum(
            norm(positions[i] - positions[j])
            for i in eachindex(positions), j in eachindex(positions) if i < j
        )

        scale = target_nn_bohr / min_dist
        positions = [scale * p for p in positions]
    end

    return positions
end

function scale_by_central_neighbours(positions; target_nn_bohr)
    centre_index = find_central_atom(positions)
    x0 = positions[centre_index]

    dists = sort([
        norm(positions[i] - x0)
        for i in eachindex(positions)
        if i != centre_index
    ])

    current_nn = mean(dists[1:12])
    scale = target_nn_bohr / current_nn

    return [scale * p for p in positions]
end

scale_by_central_neighbours (generic function with 1 method)

In [3]:
AtoBohr = 1.8897259886
a_cu_angstrom = 3.6258
cu_nn_bohr = (a_cu_angstrom / sqrt(2)) * AtoBohr

positions_cu309 = mackay_icosahedron(4; target_nn_bohr=cu_nn_bohr)

println(length(positions_cu309))

312


In [6]:
atomindex = find_central_atom(positions_cu309)

N = length(positions_cu309)
neighbours = Vector{Int}(undef, N - 1)
distances = Vector{Float64}(undef, N - 1)

cutoff = 11.338

k = find_neighbours!(neighbours, distances, positions_cu309, atomindex, cutoff)

println("Cu309 Mackay icosahedron")
println("central atomindex = ", atomindex)
println("central k = ", k)
println("central fraction = ", k/(N-1))

Cu309 Mackay icosahedron
central atomindex = 1
central k = 0
central fraction = 0.0


In [8]:
AtoBohr = 1.8897259886
a_cu_angstrom = 3.6258
cu_nn_bohr = (a_cu_angstrom / sqrt(2)) * AtoBohr

positions_cu309_raw = mackay_icosahedron(4)
positions_cu309 = scale_by_central_neighbours(
    positions_cu309_raw;
    target_nn_bohr = cu_nn_bohr
)

println(length(positions_cu309))

atomindex = find_central_atom(positions_cu309)

N = length(positions_cu309)
neighbours = Vector{Int}(undef, N - 1)
distances = Vector{Float64}(undef, N - 1)

cutoff = 11.338

k = find_neighbours!(neighbours, distances, positions_cu309, atomindex, cutoff)

println("Cu309 Mackay icosahedron")
println("central atomindex = ", atomindex)
println("central k = ", k)
println("central fraction = ", k/(N-1))

312
Cu309 Mackay icosahedron
central atomindex = 46
central k = 45
central fraction = 0.14469453376205788


In [9]:
dists = sort([
    norm(positions_cu309[i] - positions_cu309[atomindex])
    for i in eachindex(positions_cu309)
    if i != atomindex
])

println(dists[1:20])
println("max central distance = ", maximum(dists))
println("cutoff = ", cutoff)

[0.0, 0.0, 0.0, 6.459909282695508, 6.459909282695508, 6.459909282695508, 6.459909282695508, 6.4599092826955085, 6.4599092826955085, 6.4599092826955085, 6.4599092826955085, 6.4599092826955085, 6.4599092826955085, 6.4599092826955085, 6.4599092826955085, 10.990254106388045, 10.990254106388045, 10.990254106388045, 10.990254106388045, 10.990254106388045]
max central distance = 25.83963712962645
cutoff = 11.338


In [8]:
function deduplicate_positions(positions; tol=1e-8)
    seen = Set{NTuple{3,Int}}()
    unique_positions = SVector{3,Float64}[]

    for p in positions
        key = (
            round(Int, p[1] / tol),
            round(Int, p[2] / tol),
            round(Int, p[3] / tol)
        )

        if !(key in seen)
            push!(seen, key)
            push!(unique_positions, p)
        end
    end

    return unique_positions
end

function central_neighbour_distance_scale(positions; n_neigh=12, tol=1e-8)
    atomindex = find_central_atom(positions)
    x0 = positions[atomindex]

    dists = sort([
        norm(positions[i] - x0)
        for i in eachindex(positions)
        if i != atomindex && norm(positions[i] - x0) > tol
    ])

    return mean(dists[1:n_neigh])
end

central_neighbour_distance_scale (generic function with 1 method)

In [4]:
ico_55 = [[0.0000006584,       -0.0000019175,        0.0000000505],
[-0.0000005810,       -0.0000004871,        0.6678432175],
[0.1845874248,       -0.5681026047,        0.2986701538],
[-0.4832557457,       -0.3511072166,        0.2986684497],
[-0.4832557570,        0.3511046452,        0.2986669456],
[0.1845874064,        0.5681000550,        0.2986677202],
[0.5973371920,       -0.0000012681,        0.2986697030],
[-0.1845860897,       -0.5681038901,       -0.2986676192],
[-0.5973358752,       -0.0000025669,       -0.2986696020],
[-0.1845861081,        0.5680987696,       -0.2986700528],
[0.4832570624,        0.3511033815,       -0.2986683486],
[0.4832570738,       -0.3511084803,       -0.2986668445],
[0.0000018978,       -0.0000033480,       -0.6678431165],
[-0.0000017969,        0.0000009162,        1.3230014650],
[0.1871182835,       -0.5758942175,        0.9797717078],
[-0.4898861924,       -0.3559221410,       0.9797699802],
[-0.4898862039,        0.3559224872,        0.9797684555],
[0.1871182648,        0.5758945856,        0.9797692407],
[0.6055300485,        0.0000001908,        0.9797712507],
[0.7926501864,       -0.5758950093,        0.6055339635],
[0.3656681761,       -1.1254128670,        0.5916673591],
[-0.3027660545,       -0.9318173412,        0.6055326929],
[-0.9573332453,       -0.6955436707,        0.5916639831],
[-0.9797705418,       -0.0000006364,        0.6055294407],
[-0.9573332679,        0.6955423392,        0.5916610035],
[-0.3027660847,        0.9318160902,        0.6055287012],
[0.3656681396,        1.1254115783,        0.5916625380],
[0.7926501677,        0.5758937939,        0.6055314964],
[1.1833279992,       -0.0000006311,        0.5916664660],
[0.6770051458,       -0.9318186223,        0.0000033028],
[0.0000006771,       -1.1517907207,        0.0000025175],
[-0.6770037988,       -0.9318186442,        0.0000007900],
[-1.0954155825,       -0.3559242494,       -0.0000012200],
[-1.0954155940,        0.3559203788,       -0.0000027447],
[-0.6770038290,        0.9318147872,       -0.0000032017],
[0.0000006397,        1.1517868856,       -0.0000024165],
[0.6770051155,        0.9318148091,       -0.0000006889],
[1.0954168993,        0.3559204143,        0.0000013211],
[1.0954169108,       -0.3559242139,        0.0000028458],
[0.3027674014,       -0.9318199253,       -0.6055286002],
[-0.3656668229,       -1.1254154134,       -0.5916624370],
[-0.7926488510,       -0.5758976290,       -0.6055313954],
[-1.1833266824,       -0.0000032040,       -0.5916663649],
[-0.7926488697,        0.5758911742,       -0.6055338624],
[-0.3656668594,        1.1254090319,       -0.5916672580],
[0.3027673712,        0.9318135061,       -0.6055325919],
[0.9573345621,        0.6955398357,       -0.5916638820],
[0.9797718586,       -0.0000031986,       -0.6055293396],
[0.9573345846,       -0.6955461743,       -0.5916609025],
[-0.1871169480,       -0.5758984207,       -0.9797691397],
[-0.6055287318,       -0.0000040259,       -0.9797711497],
[-0.1871169667,        0.5758903824,       -0.9797716067],
[0.4898875091,        0.3559183059,       -0.9797698792],
[0.4898875207,       -0.3559263223,       -0.9797683545],
[0.0000031136,       -0.0000047513,       -1.3230013639]]

#convert to Bohr
nmtobohr = 18.8973

copperconstant = 0.36258*nmtobohr

pos_cu55 = copperconstant*ico_55

positions_cu55 = [SVector{3,Float64}(p) for p in pos_cu55]

positions_cu309_raw = mackay_icosahedron(4)

positions_cu309_raw = deduplicate_positions(positions_cu309_raw)

println("raw unique count = ", length(positions_cu309_raw))

target_nn = central_neighbour_distance_scale(positions_cu55)
raw_nn = central_neighbour_distance_scale(positions_cu309_raw)

scale = target_nn / raw_nn

positions_cu309 = [scale * p for p in positions_cu309_raw]

println("scaled count = ", length(positions_cu309))
println("target Cu55 central nearest-neighbour distance = ", target_nn)
println("Cu309 central nearest-neighbour distance after scaling = ", central_neighbour_distance_scale(positions_cu309))

raw unique count = 309


UndefVarError: UndefVarError: `find_central_atom` not defined

In [12]:
atomindex = find_central_atom(positions_cu309)

N = length(positions_cu309)
neighbours = Vector{Int}(undef, N - 1)
distances = Vector{Float64}(undef, N - 1)

cutoff = 11.338

k = find_neighbours!(neighbours, distances, positions_cu309, atomindex, cutoff)

println("Cu309 Mackay icosahedron")
println("central atomindex = ", atomindex)
println("central k = ", k)
println("central fraction = ", k / (N - 1))

dists = sort([
    norm(positions_cu309[i] - positions_cu309[atomindex])
    for i in eachindex(positions_cu309)
    if i != atomindex && norm(positions_cu309[i] - positions_cu309[atomindex]) > 1e-8
])

println("first 20 central distances = ", dists[1:20])
println("max central distance = ", maximum(dists))
println("cutoff = ", cutoff)

Cu309 Mackay icosahedron
central atomindex = 46
central k = 74
central fraction = 0.24025974025974026
first 20 central distances = [4.575916480942233, 4.575916480942233, 4.575916480942233, 4.575916480942233, 4.575916480942233, 4.575916480942233, 4.575916480942233, 4.575916480942233, 4.575916480942233, 4.575916480942233, 4.575916480942233, 4.575916480942233, 7.7850141069132075, 7.7850141069132075, 7.7850141069132075, 7.7850141069132075, 7.7850141069132075, 7.7850141069132075, 7.7850141069132075, 7.7850141069132075]
max central distance = 18.30366592295037
cutoff = 11.338


In [12]:
for shell in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
    positions_raw = deduplicate_positions(mackay_icosahedron(shell))

    target_nn = central_neighbour_distance_scale(positions_cu55)
    raw_nn = central_neighbour_distance_scale(positions_raw)
    scale = target_nn / raw_nn

    positions_mackay = [scale * p for p in positions_raw]

    N = length(positions_mackay)
    atomindex = find_central_atom(positions_mackay)

    neighbours = Vector{Int}(undef, N - 1)
    distances = Vector{Float64}(undef, N - 1)

    cutoff = 11.338

    central_k = find_neighbours!(
        neighbours,
        distances,
        positions_mackay,
        atomindex,
        cutoff
    )

    ks = Int[]

    for i in eachindex(positions_mackay)
        k_i = find_neighbours!(
            neighbours,
            distances,
            positions_mackay,
            i,
            cutoff
        )
        push!(ks, k_i)
    end

    println()
    println("Mackay shell = ", shell)
    println("N = ", N)
    println("central atomindex = ", atomindex)
    println("central k = ", central_k)
    println("central fraction = ", central_k / (N - 1))
    println("min k = ", minimum(ks))
    println("max k = ", maximum(ks))
    println("mean k = ", mean(ks))
    println("mean fraction = ", mean(ks) / (N - 1))
end


Mackay shell = 2
N = 55
central atomindex = 16
central k = 54
central fraction = 1.0
min k = 22
max k = 54
mean k = 32.07272727272727
mean fraction = 0.5939393939393939

Mackay shell = 3
N = 147
central atomindex = 33
central k = 74
central fraction = 0.5068493150684932
min k = 22
max k = 74
mean k = 44.435374149659864
mean fraction = 0.30435187773739636

Mackay shell = 4
N = 309
central atomindex = 46
central k = 74
central fraction = 0.24025974025974026
min k = 22
max k = 80
mean k = 51.97411003236246
mean fraction = 0.16874711049468333

Mackay shell = 5
N = 561
central atomindex = 79
central k = 74
central fraction = 0.13214285714285715
min k = 22
max k = 82
mean k = 56.948306595365416
mean fraction = 0.1016934046345811

Mackay shell = 6
N = 923
central atomindex = 416
central k = 74
central fraction = 0.08026030368763558
min k = 22
max k = 83
mean k = 60.45070422535211
mean fraction = 0.06556475512511076

Mackay shell = 7
N = 1415
central atomindex = 664
central k = 74
central fra

Next, we test the calculation of the angular symmetry function values via calculation over all unordered tripplets that include the moved atom, and compare that to the 

In [13]:
function neighbour_union_for_move(old_positions, new_positions, atomindex, cutoff)
    N = length(old_positions)

    old_neigh = Vector{Int}(undef, N - 1)
    old_dist = Vector{Float64}(undef, N - 1)

    new_neigh = Vector{Int}(undef, N - 1)
    new_dist = Vector{Float64}(undef, N - 1)

    k_old = find_neighbours!(old_neigh, old_dist, old_positions, atomindex, cutoff)
    k_new = find_neighbours!(new_neigh, new_dist, new_positions, atomindex, cutoff)

    return union(old_neigh[1:k_old], new_neigh[1:k_new])
end

#=

function radial_symmetry_calculation_neigh!(
    g_vector,
    atomindex,
    neighbour_indices,
    dist2_mat,
    new_dist2_vector,
    f_matrix,
    new_f_vector,
    symmetry_function
)
    if symmetry_function.type_vec == Int(11)
        η = symmetry_function.eta
        g_norm = symmetry_function.G_norm

        for index2 in neighbour_indices
            ParallelTemperingMonteCarlo.DeltaMatrix.calc_new_symmetry_value!(
                g_vector,
                atomindex,
                index2,
                dist2_mat,
                new_dist2_vector,
                f_matrix,
                new_f_vector,
                η,
                g_norm
            )
        end
    end

    return g_vector
end




angular_symmetry_calculation_neigh!(
    g_vector,
    atomindex,
    neighbour_indices,
    newposition,
    positions,
    dist2_mat,
    new_dist2_vector,
    f_matrix,
    new_f_vector,
    symmetry_function
)
    if symmetry_function.type_vec == Int(111)

        η = symmetry_function.eta
        λ = symmetry_function.lambda
        ζ = symmetry_function.zeta
        tpz = symmetry_function.tpz

        for a in 1:length(neighbour_indices)-1
            j_index = neighbour_indices[a]

            for b in a+1:length(neighbour_indices)
                k_index = neighbour_indices[b]

                # If j-k is outside the cutoff, both old and new angular products are zero.
                if f_matrix[j_index, k_index] != 0.0
                    g_vector = ParallelTemperingMonteCarlo.DeltaMatrix.calc_new_symmetry_value!(
                        g_vector,
                        atomindex,
                        j_index,
                        k_index,
                        newposition,
                        positions,
                        dist2_mat,
                        new_dist2_vector,
                        f_matrix,
                        new_f_vector,
                        η,
                        λ,
                        ζ,
                        tpz
                    )
                end
            end
        end
    end

    return g_vector
end


function total_symm_neigh!(
    g_matrix,
    positions,
    new_position,
    dist2_mat,
    new_dist2_vector,
    f_matrix,
    new_f_vector,
    atomindex,
    neighbour_indices,
    radsymmfunctions,
    angsymmfunctions,
    Nrad,
    Nang
)
    for g_index in 1:Nrad
        g_matrix[g_index, :] = radial_symmetry_calculation_neigh!(
            g_matrix[g_index, :],
            atomindex,
            neighbour_indices,
            dist2_mat,
            new_dist2_vector,
            f_matrix,
            new_f_vector,
            radsymmfunctions[g_index]
        )
    end

    for g_index in 1:Nang
        true_index = g_index + Nrad

        g_matrix[true_index, :] = angular_symmetry_calculation_neigh!(
            g_matrix[true_index, :],
            atomindex,
            neighbour_indices,
            new_position,
            positions,
            dist2_mat,
            new_dist2_vector,
            f_matrix,
            new_f_vector,
            angsymmfunctions[g_index]
        )
    end

    return g_matrix
end

=#

neighbour_union_for_move (generic function with 1 method)

In [10]:

#=
using BenchmarkTools
using StaticArrays
using Statistics


# Use your Cu147 positions here
positions = positions_m147

N = length(positions)
cutoff = 11.338

atomindex = find_central_atom(positions)
displacement = SVector{3,Float64}(0.05, 0.0, 0.0)

new_positions = copy(positions)
new_positions[atomindex] = positions[atomindex] + displacement
new_position = new_positions[atomindex]

dist2_mat = find_distance2_mat(positions)
f_matrix = cutoff_function.(sqrt.(dist2_mat), Ref(cutoff))

new_dist2_vec = [
    sum((new_position .- positions[j]).^2)
    for j in eachindex(positions)
]
new_dist2_vec[atomindex] = 0.0

new_f_vec = cutoff_function.(sqrt.(new_dist2_vec), Ref(cutoff))

neighbour_indices = neighbour_union_for_move(
    positions,
    new_positions,
    atomindex,
    cutoff
)

println("N = ", N)
println("atomindex = ", atomindex)
println("neighbour count = ", length(neighbour_indices))
println("neighbour fraction = ", length(neighbour_indices)/(N-1))

=#

using BenchmarkTools
using StaticArrays
using LinearAlgebra
using Statistics

# ------------------------------------------------------------
# Generate Cu147 Mackay icosahedron, scaled using project Cu55
# ------------------------------------------------------------

positions_m147_raw = deduplicate_positions(mackay_icosahedron(3))

target_nn = central_neighbour_distance_scale(positions_cu55)
raw_nn = central_neighbour_distance_scale(positions_m147_raw)

scale = target_nn / raw_nn

positions = [scale * p for p in positions_m147_raw]

println("Generated Mackay icosahedron")
println("N = ", length(positions))
println("target Cu55 central nearest-neighbour distance = ", target_nn)
println("generated central nearest-neighbour distance = ", central_neighbour_distance_scale(positions))
println()

# ------------------------------------------------------------
# Artificial atom move
# ------------------------------------------------------------

N = length(positions)
cutoff = 11.338

atomindex = find_central_atom(positions)
#displacement = SVector{3,Float64}(0.05, 0.0, 0.0)
displacement = SVector{3,Float64}(0.05, 0.0, 0.0)

new_positions = copy(positions)
new_positions[atomindex] = positions[atomindex] + displacement
new_position = new_positions[atomindex]

# ------------------------------------------------------------
# Old and new distance/cutoff data
# ------------------------------------------------------------

dist2_mat = find_distance2_mat(positions)
f_matrix = cutoff_function.(sqrt.(dist2_mat), Ref(cutoff))

new_dist2_vec = [
    sum((new_position .- positions[j]).^2)
    for j in eachindex(positions)
]
new_dist2_vec[atomindex] = 0.0

new_f_vec = cutoff_function.(sqrt.(new_dist2_vec), Ref(cutoff))

# ------------------------------------------------------------
# Neighbour bookkeeping
# ------------------------------------------------------------

old_neigh_buffer = Vector{Int}(undef, N - 1)
old_dist_buffer = Vector{Float64}(undef, N - 1)

new_neigh_buffer = Vector{Int}(undef, N - 1)
new_dist_buffer = Vector{Float64}(undef, N - 1)

k_old = find_neighbours!(
    old_neigh_buffer,
    old_dist_buffer,
    positions,
    atomindex,
    cutoff
)

k_new = find_neighbours!(
    new_neigh_buffer,
    new_dist_buffer,
    new_positions,
    atomindex,
    cutoff
)

old_neigh = sort(old_neigh_buffer[1:k_old])
new_neigh = sort(new_neigh_buffer[1:k_new])

entered = setdiff(new_neigh, old_neigh)
left = setdiff(old_neigh, new_neigh)
stayed = intersect(old_neigh, new_neigh)

neighbour_indices = sort(union(old_neigh, new_neigh))

println("Move and neighbour bookkeeping")
println("atomindex = ", atomindex)
println("displacement = ", displacement)
println("cutoff = ", cutoff)
println()

println("old neighbour count = ", length(old_neigh))
println("new neighbour count = ", length(new_neigh))
println("union neighbour count = ", length(neighbour_indices))
println("old neighbour fraction = ", length(old_neigh)/(N-1))
println("new neighbour fraction = ", length(new_neigh)/(N-1))
println("union neighbour fraction = ", length(neighbour_indices)/(N-1))
println()

println("old = ", old_neigh)
println("new = ", new_neigh)
println("entered = ", entered)
println("left = ", left)
println("stayed = ", stayed)

Generated Mackay icosahedron
N = 147
target Cu55 central nearest-neighbour distance = 4.575916480942233
generated central nearest-neighbour distance = 4.575916480942233

Move and neighbour bookkeeping
atomindex = 33
displacement = [0.05, 0.0, 0.0]
cutoff = 11.338

old neighbour count = 74
new neighbour count = 74
union neighbour count = 74
old neighbour fraction = 0.5068493150684932
new neighbour fraction = 0.5068493150684932
union neighbour fraction = 0.5068493150684932

old = [5, 9, 10, 11, 12, 15, 17, 21, 22, 23, 28, 35, 36, 37, 38, 39, 42, 45, 46, 47, 48, 49, 58, 59, 62, 64, 65, 66, 67, 68, 69, 70, 71, 73, 74, 75, 76, 77, 79, 80, 81, 87, 89, 92, 94, 96, 97, 98, 101, 102, 104, 105, 106, 107, 110, 111, 112, 116, 117, 118, 119, 120, 121, 123, 124, 125, 128, 129, 134, 138, 139, 144, 145, 147]
new = [5, 9, 10, 11, 12, 15, 17, 21, 22, 23, 28, 35, 36, 37, 38, 39, 42, 45, 46, 47, 48, 49, 58, 59, 62, 64, 65, 66, 67, 68, 69, 70, 71, 73, 74, 75, 76, 77, 79, 80, 81, 87, 89, 92, 94, 96, 97, 98,

In [14]:
# Recompute full symmetry matrix for the current 147-atom positions

dist2_mat = find_distance2_mat(positions)
f_matrix = cutoff_function.(sqrt.(dist2_mat), Ref(pot.r_cut))

nrad = length(pot.radsymfunctions)
nang = length(pot.angsymfunctions)

G_full = total_symm_calc(
    positions,
    dist2_mat,
    f_matrix,
    pot.radsymfunctions,
    pot.angsymfunctions,
    nrad,
    nang
)

println("size(G_full) = ", size(G_full))

G_current = copy(G_full)
G_neigh = copy(G_full)

total_symm!(
    G_current,
    positions,
    new_position,
    dist2_mat,
    new_dist2_vec,
    f_matrix,
    new_f_vec,
    atomindex,
    pot.radsymfunctions,
    pot.angsymfunctions,
    nrad,
    nang
)

total_symm_neigh!(
    G_neigh,
    positions,
    new_position,
    dist2_mat,
    new_dist2_vec,
    f_matrix,
    new_f_vec,
    atomindex,
    neighbour_indices,
    pot.radsymfunctions,
    pot.angsymfunctions,
    nrad,
    nang
)

println("maximum difference = ", maximum(abs.(G_current .- G_neigh)))

size(G_full) = (88, 147)
maximum difference = 0.0


In [15]:
println("change caused by move to G = ", maximum(abs.(G_current .- G_full)))
println("method difference for G = ", maximum(abs.(G_current .- G_neigh)))
println("relative method error = ",
    maximum(abs.(G_current .- G_neigh)) / maximum(abs.(G_current .- G_full))
)

change caused by move to G = 26.722975877039463
method difference for G = 0.0
relative method error = 0.0


In [11]:
#= @btime total_symm!(
    $G_current,
    $positions,
    $new_position,
    $dist2_mat,
    $new_dist2_vec,
    $f_matrix,
    $new_f_vec,
    $atomindex,
    $(pot.radsymfunctions),
    $(pot.angsymfunctions),
    $nrad,
    $nang
)

@btime total_symm_neigh!(
    $G_neigh,
    $positions,
    $new_position,
    $dist2_mat,
    $new_dist2_vec,
    $f_matrix,
    $new_f_vec,
    $atomindex,
    $neighbour_indices,
    $(pot.radsymfunctions),
    $(pot.angsymfunctions),
    $nrad,
    $nang
)
    =#
@btime begin
    G_current = copy($G_full)

    total_symm!(
        G_current,
        $positions,
        $new_position,
        $dist2_mat,
        $new_dist2_vec,
        $f_matrix,
        $new_f_vec,
        $atomindex,
        $(pot.radsymfunctions),
        $(pot.angsymfunctions),
        $nrad,
        $nang
    )
end

@btime begin
    G_neigh = copy($G_full)

    total_symm_neigh!(
        G_neigh,
        $positions,
        $new_position,
        $dist2_mat,
        $new_dist2_vec,
        $f_matrix,
        $new_f_vec,
        $atomindex,
        $neighbour_indices,
        $(pot.radsymfunctions),
        $(pot.angsymfunctions),
        $nrad,
        $nang
    )
end
    

LoadError: UndefVarError: `G_full` not defined

In [ ]:


function neighbour_union_from_cutoff_vectors(f_old_row, f_new_vec, atomindex)
    neighbours = Int[]

    for j in eachindex(f_new_vec)
        if j != atomindex && (f_old_row[j] != 0.0 || f_new_vec[j] != 0.0)
            push!(neighbours, j)
        end
    end

    return neighbours
end


#=
neighbour_indices_from_f = neighbour_union_from_cutoff_vectors(
    f_matrix[atomindex, :],
    new_f_vec,
    atomindex
)

println("distance-based union count = ", length(neighbour_indices))
println("f-vector union count = ", length(neighbour_indices_from_f))
println("same union? ", neighbour_indices == neighbour_indices_from_f)

println("in distance union but not f union = ", setdiff(neighbour_indices, neighbour_indices_from_f))
println("in f union but not distance union = ", setdiff(neighbour_indices_from_f, neighbour_indices))
=#

neighbour_union_from_cutoff_vectors (generic function with 1 method)

Testing the new code that runs a full run of the neighbour restricted Symmetry matrix calculation and then the energy calculation to compare against the currently PTMCMC implemented versions of these for accuracy and runtime.

In [5]:
using Pkg
Pkg.activate("/home/rev/ParallelTemperingMonteCarlo.jl")

using ParallelTemperingMonteCarlo
using StaticArrays
using LinearAlgebra
using DelimitedFiles
using BenchmarkTools
using Statistics

# ------------------------------------------------------------
# Helper functions for Mackay icosahedron generation
# ------------------------------------------------------------

function mackay_icosahedron(shell::Int)
    φ = (1 + sqrt(5.0)) / 2

    verts = SVector{3,Float64}[]

    for s1 in (-1.0, 1.0), s2 in (-1.0, 1.0)
        push!(verts, SVector(0.0, s1, s2 * φ))
        push!(verts, SVector(s1, s2 * φ, 0.0))
        push!(verts, SVector(s1 * φ, 0.0, s2))
    end

    edge = minimum(norm(verts[i] - verts[j])
                   for i in eachindex(verts), j in eachindex(verts) if i < j)

    faces = Tuple{Int,Int,Int}[]

    for i in 1:length(verts)-2
        for j in i+1:length(verts)-1
            for k in j+1:length(verts)
                if abs(norm(verts[i] - verts[j]) - edge) < 1e-8 &&
                   abs(norm(verts[i] - verts[k]) - edge) < 1e-8 &&
                   abs(norm(verts[j] - verts[k]) - edge) < 1e-8
                    push!(faces, (i, j, k))
                end
            end
        end
    end

    pointset = Set{NTuple{3,Float64}}()

    for (a, b, c) in faces
        v1, v2, v3 = verts[a], verts[b], verts[c]

        for i in 0:shell
            for j in 0:(shell - i)
                for k in 0:(shell - i - j)
                    p = (i*v1 + j*v2 + k*v3) / shell
                    push!(pointset, (
                        round(p[1], digits=10),
                        round(p[2], digits=10),
                        round(p[3], digits=10)
                    ))
                end
            end
        end
    end

    positions = [SVector{3,Float64}(p) for p in pointset]
    centre = sum(positions) / length(positions)

    return [p - centre for p in positions]
end

function deduplicate_positions(positions; tol=1e-8)
    seen = Set{NTuple{3,Int}}()
    unique_positions = SVector{3,Float64}[]

    for p in positions
        key = (
            round(Int, p[1] / tol),
            round(Int, p[2] / tol),
            round(Int, p[3] / tol)
        )

        if !(key in seen)
            push!(seen, key)
            push!(unique_positions, p)
        end
    end

    return unique_positions
end

function find_central_atom(positions)
    return argmin([norm(p) for p in positions])
end

function central_neighbour_distance_scale(positions; n_neigh=12, tol=1e-8)
    atomindex = find_central_atom(positions)
    x0 = positions[atomindex]

    dists = sort([
        norm(positions[i] - x0)
        for i in eachindex(positions)
        if i != atomindex && norm(positions[i] - x0) > tol
    ])

    return mean(dists[1:n_neigh])
end

# ------------------------------------------------------------
# Build RuNNer potential
# ------------------------------------------------------------

data_path = "/home/rev/ParallelTemperingMonteCarlo.jl/scripts/data"

X = [ 11 0.001 0.000 11.338
      10 0.001 0.000 11.338
      11 0.020 0.000 11.338
      10 0.020 0.000 11.338
      11 0.035 0.000 11.338
      10 0.035 0.000 11.338
      11 0.100 0.000 11.338
      10 0.100 0.000 11.338
      11 0.400 0.000 11.338
      10 0.400 0.000 11.338 ]

V = [[0.0001,1,1,11.338],[0.0001,-1,2,11.338],
     [0.003,-1,1,11.338],[0.003,-1,2,11.338],
     [0.008,-1,1,11.338],[0.008,-1,2,11.338],[0.008,1,2,11.338],
     [0.015,1,1,11.338],[0.015,-1,2,11.338],[0.015,-1,4,11.338],
     [0.015,-1,16,11.338],[0.025,-1,1,11.338],[0.025,1,1,11.338],
     [0.025,1,2,11.338],[0.025,-1,4,11.338],[0.025,-1,16,11.338],
     [0.025,1,16,11.338],[0.045,1,1,11.338],[0.045,-1,2,11.338],
     [0.045,-1,4,11.338],[0.045,1,4,11.338],[0.045,1,16,11.338],
     [0.08,1,1,11.338],[0.08,-1,2,11.338],
     [0.08,-1,4,11.338],[0.08,1,4,11.338]]

T = [111, 110, 100]

scalingvalues = readdlm(joinpath(data_path, "scaling.data"))
G_value_vec = []

for row in eachrow(scalingvalues[1:88, :])
    push!(G_value_vec, [row[4], row[3]])
end

radsymmvec = RadialType2{Float64}[]
angularsymmvec = AngularType3{Float64}[]

for symmindex in eachindex(eachrow(X))
    row = X[symmindex, :]
    push!(radsymmvec, RadialType2{Float64}(row[2], row[4], Int(row[1]), G_value_vec[symmindex]))
end

let n_index = 10
    for element in V
        for types in T
            n_index += 1
            push!(angularsymmvec,
                AngularType3{Float64}(element[1], element[2], element[3], 11.338, types, G_value_vec[n_index])
            )
        end
    end
end

num_nodes = Int32[88, 20, 20, 1]
activation_functions = Int32[1, 2, 2, 1]
weights = vec(readdlm(joinpath(data_path, "weights.029.data")))

nnp = NeuralNetworkPotential(num_nodes, activation_functions, weights)
pot = RuNNerPotential(nnp, radsymmvec, angularsymmvec)

# ------------------------------------------------------------
# Generate Cu147 Mackay cluster
# ------------------------------------------------------------

target_nn_bohr = 4.575916480942233  # from project Cu55 central nearest-neighbour spacing

positions_raw = deduplicate_positions(mackay_icosahedron(3))
raw_nn = central_neighbour_distance_scale(positions_raw)
positions = [(target_nn_bohr / raw_nn) * p for p in positions_raw]

N = length(positions)

println("Generated Cu147")
println("N = ", N)
println("central NN distance = ", central_neighbour_distance_scale(positions))

cluster_radius = maximum(norm(p) for p in positions)
bc = SphericalBC(radius = cluster_radius + 5.0)
config = Config(positions, bc)

# ------------------------------------------------------------
# Initialise MCState and trial move
# ------------------------------------------------------------

ensemble = NVT(N)

temp = 1000.0
beta = 1.0 / temp

mc_state = MCState(temp, beta, config, ensemble, pot)

atomindex = find_central_atom(config.pos)
displacement = SVector{3,Float64}(0.05, 0.0, 0.0)
trial_move = config.pos[atomindex] + displacement

new_dist2_vec = [
    sum((trial_move .- config.pos[j]).^2)
    for j in eachindex(config.pos)
]
new_dist2_vec[atomindex] = 0.0

mc_state.ensemble_variables.index = atomindex
mc_state.ensemble_variables.trial_move = trial_move

# ------------------------------------------------------------
# Neighbour-restricted state and energy update functions
# ------------------------------------------------------------
#=
function get_new_state_vars_neigh!(
    trial_pos,
    atomindex,
    config,
    potential_variables,
    dist2_mat,
    new_dist2_vec,
    pot
)
    Nrad = length(pot.radsymfunctions)
    Nang = length(pot.angsymfunctions)

    potential_variables.new_f_vec = cutoff_function.(sqrt.(new_dist2_vec), Ref(pot.r_cut))

    neighbour_indices_from_f = neighbour_union_from_cutoff_vectors(
        potential_variables.f_matrix[atomindex, :],
        potential_variables.new_f_vec,
        atomindex
    )

    potential_variables.new_g_matrix = copy(potential_variables.g_matrix)

    potential_variables.new_g_matrix = total_symm_neigh!(
        potential_variables.new_g_matrix,
        config.pos,
        trial_pos,
        dist2_mat,
        new_dist2_vec,
        potential_variables.f_matrix,
        potential_variables.new_f_vec,
        atomindex,
        neighbour_indices_from_f,
        pot.radsymfunctions,
        pot.angsymfunctions,
        Nrad,
        Nang
    )

    return potential_variables
end

function energy_update_neigh!(
    ensemblevariables,
    config,
    potential_variables,
    dist2_mat,
    new_dist2_vec,
    en_tot,
    pot
)
    if any(new_dist2_vec[i] < pot.boundary for i in eachindex(new_dist2_vec) if i != ensemblevariables.index)
        new_en = 100.0
    else
        potential_variables = get_new_state_vars_neigh!(
            ensemblevariables.trial_move,
            ensemblevariables.index,
            config,
            potential_variables,
            dist2_mat,
            new_dist2_vec,
            pot
        )

        potential_variables, new_en = calc_new_runner_energy!(potential_variables, pot)
    end

    return potential_variables, new_en
end
=#
# ------------------------------------------------------------
# Correctness comparison
# ------------------------------------------------------------

pv_old = deepcopy(mc_state.potential_variables)
pv_neigh = deepcopy(mc_state.potential_variables)

pv_old, en_old = energy_update!(
    mc_state.ensemble_variables,
    mc_state.config,
    pv_old,
    mc_state.dist2_mat,
    new_dist2_vec,
    mc_state.en_tot,
    pot
)

pv_neigh, en_neigh = energy_update_neigh!(
    mc_state.ensemble_variables,
    mc_state.config,
    pv_neigh,
    mc_state.dist2_mat,
    new_dist2_vec,
    mc_state.en_tot,
    pot
)

println()
println("Correctness test")
println("energy old = ", en_old)
println("energy neigh = ", en_neigh)
println("energy difference = ", abs(en_old - en_neigh))
println("G difference = ", maximum(abs.(pv_old.new_g_matrix .- pv_neigh.new_g_matrix)))
println("new atom-energy vector difference = ", maximum(abs.(pv_old.new_en_atom .- pv_neigh.new_en_atom)))

# ------------------------------------------------------------
# Benchmark comparison
# ------------------------------------------------------------

println()
println("Benchmarking old energy_update! path")
@btime energy_update!(
    $(mc_state.ensemble_variables),
    $(mc_state.config),
    deepcopy($(mc_state.potential_variables)),
    $(mc_state.dist2_mat),
    $new_dist2_vec,
    $(mc_state.en_tot),
    $pot
);

println()
println("Benchmarking neighbour-restricted energy_update_neigh! path")
@btime energy_update_neigh!(
    $(mc_state.ensemble_variables),
    $(mc_state.config),
    deepcopy($(mc_state.potential_variables)),
    $(mc_state.dist2_mat),
    $new_dist2_vec,
    $(mc_state.en_tot),
    $pot
);

  Activating project at `~/ParallelTemperingMonteCarlo.jl`


Generated Cu147
N = 147
central NN distance = 4.575916480942233

Correctness test
energy old = -22.428406923912934
energy neigh = -22.428406923912934
energy difference = 0.0
G difference = 0.0
new atom-energy vector difference = 0.0

Benchmarking old energy_update! path
  47.697 ms (117 allocations: 587.38 KiB)

Benchmarking neighbour-restricted energy_update_neigh! path
  8.686 ms (110 allocations: 589.16 KiB)


In [ ]:
using Pkg
Pkg.activate("/home/rev/ParallelTemperingMonteCarlo.jl")

using ParallelTemperingMonteCarlo
using StaticArrays
using LinearAlgebra
using DelimitedFiles
using BenchmarkTools
using Statistics

# ------------------------------------------------------------
# Helper functions for Mackay icosahedron generation
# ------------------------------------------------------------

function mackay_icosahedron(shell::Int)
    φ = (1 + sqrt(5.0)) / 2

    verts = SVector{3,Float64}[]

    for s1 in (-1.0, 1.0), s2 in (-1.0, 1.0)
        push!(verts, SVector(0.0, s1, s2 * φ))
        push!(verts, SVector(s1, s2 * φ, 0.0))
        push!(verts, SVector(s1 * φ, 0.0, s2))
    end

    edge = minimum(norm(verts[i] - verts[j])
                   for i in eachindex(verts), j in eachindex(verts) if i < j)

    faces = Tuple{Int,Int,Int}[]

    for i in 1:length(verts)-2
        for j in i+1:length(verts)-1
            for k in j+1:length(verts)
                if abs(norm(verts[i] - verts[j]) - edge) < 1e-8 &&
                   abs(norm(verts[i] - verts[k]) - edge) < 1e-8 &&
                   abs(norm(verts[j] - verts[k]) - edge) < 1e-8
                    push!(faces, (i, j, k))
                end
            end
        end
    end

    pointset = Set{NTuple{3,Float64}}()

    for (a, b, c) in faces
        v1, v2, v3 = verts[a], verts[b], verts[c]

        for i in 0:shell
            for j in 0:(shell - i)
                for k in 0:(shell - i - j)
                    p = (i*v1 + j*v2 + k*v3) / shell
                    push!(pointset, (
                        round(p[1], digits=10),
                        round(p[2], digits=10),
                        round(p[3], digits=10)
                    ))
                end
            end
        end
    end

    positions = [SVector{3,Float64}(p) for p in pointset]
    centre = sum(positions) / length(positions)

    return [p - centre for p in positions]
end

function deduplicate_positions(positions; tol=1e-8)
    seen = Set{NTuple{3,Int}}()
    unique_positions = SVector{3,Float64}[]

    for p in positions
        key = (
            round(Int, p[1] / tol),
            round(Int, p[2] / tol),
            round(Int, p[3] / tol)
        )

        if !(key in seen)
            push!(seen, key)
            push!(unique_positions, p)
        end
    end

    return unique_positions
end

function find_central_atom(positions)
    return argmin([norm(p) for p in positions])
end

function central_neighbour_distance_scale(positions; n_neigh=12, tol=1e-8)
    atomindex = find_central_atom(positions)
    x0 = positions[atomindex]

    dists = sort([
        norm(positions[i] - x0)
        for i in eachindex(positions)
        if i != atomindex && norm(positions[i] - x0) > tol
    ])

    return mean(dists[1:n_neigh])
end

# ------------------------------------------------------------
# Build RuNNer potential
# ------------------------------------------------------------

data_path = "/home/rev/ParallelTemperingMonteCarlo.jl/scripts/data"

X = [ 11 0.001 0.000 11.338
      10 0.001 0.000 11.338
      11 0.020 0.000 11.338
      10 0.020 0.000 11.338
      11 0.035 0.000 11.338
      10 0.035 0.000 11.338
      11 0.100 0.000 11.338
      10 0.100 0.000 11.338
      11 0.400 0.000 11.338
      10 0.400 0.000 11.338 ]

V = [[0.0001,1,1,11.338],[0.0001,-1,2,11.338],
     [0.003,-1,1,11.338],[0.003,-1,2,11.338],
     [0.008,-1,1,11.338],[0.008,-1,2,11.338],[0.008,1,2,11.338],
     [0.015,1,1,11.338],[0.015,-1,2,11.338],[0.015,-1,4,11.338],
     [0.015,-1,16,11.338],[0.025,-1,1,11.338],[0.025,1,1,11.338],
     [0.025,1,2,11.338],[0.025,-1,4,11.338],[0.025,-1,16,11.338],
     [0.025,1,16,11.338],[0.045,1,1,11.338],[0.045,-1,2,11.338],
     [0.045,-1,4,11.338],[0.045,1,4,11.338],[0.045,1,16,11.338],
     [0.08,1,1,11.338],[0.08,-1,2,11.338],
     [0.08,-1,4,11.338],[0.08,1,4,11.338]]

T = [111, 110, 100]

scalingvalues = readdlm(joinpath(data_path, "scaling.data"))
G_value_vec = []

for row in eachrow(scalingvalues[1:88, :])
    push!(G_value_vec, [row[4], row[3]])
end

radsymmvec = RadialType2{Float64}[]
angularsymmvec = AngularType3{Float64}[]

for symmindex in eachindex(eachrow(X))
    row = X[symmindex, :]
    push!(radsymmvec, RadialType2{Float64}(row[2], row[4], Int(row[1]), G_value_vec[symmindex]))
end

let n_index = 10
    for element in V
        for types in T
            n_index += 1
            push!(angularsymmvec,
                AngularType3{Float64}(element[1], element[2], element[3], 11.338, types, G_value_vec[n_index])
            )
        end
    end
end

num_nodes = Int32[88, 20, 20, 1]
activation_functions = Int32[1, 2, 2, 1]
weights = vec(readdlm(joinpath(data_path, "weights.029.data")))

nnp = NeuralNetworkPotential(num_nodes, activation_functions, weights)
pot = RuNNerPotential(nnp, radsymmvec, angularsymmvec)

# ------------------------------------------------------------
# Generate Cu147 Mackay cluster
# ------------------------------------------------------------

target_nn_bohr = 4.575916480942233  # from project Cu55 central nearest-neighbour spacing

positions_raw = deduplicate_positions(mackay_icosahedron(3))
raw_nn = central_neighbour_distance_scale(positions_raw)
positions = [(target_nn_bohr / raw_nn) * p for p in positions_raw]

N = length(positions)

println("Generated Cu147")
println("N = ", N)
println("central NN distance = ", central_neighbour_distance_scale(positions))

cluster_radius = maximum(norm(p) for p in positions)
bc = SphericalBC(radius = cluster_radius + 5.0)
config = Config(positions, bc)

# ------------------------------------------------------------
# Initialise MCState and trial move
# ------------------------------------------------------------

ensemble = NVT(N)

temp = 1000.0
beta = 1.0 / temp

mc_state = MCState(temp, beta, config, ensemble, pot)

atomindex = find_central_atom(config.pos)
displacement = SVector{3,Float64}(0.05, 0.0, 0.0)
trial_move = config.pos[atomindex] + displacement

new_dist2_vec = [
    sum((trial_move .- config.pos[j]).^2)
    for j in eachindex(config.pos)
]
new_dist2_vec[atomindex] = 0.0

mc_state.ensemble_variables.index = atomindex
mc_state.ensemble_variables.trial_move = trial_move

# ------------------------------------------------------------
# Neighbour-restricted state and energy update functions
# ------------------------------------------------------------
#=
function get_new_state_vars_neigh!(
    trial_pos,
    atomindex,
    config,
    potential_variables,
    dist2_mat,
    new_dist2_vec,
    pot
)
    Nrad = length(pot.radsymfunctions)
    Nang = length(pot.angsymfunctions)

    potential_variables.new_f_vec = cutoff_function.(sqrt.(new_dist2_vec), Ref(pot.r_cut))

    neighbour_indices_from_f = neighbour_union_from_cutoff_vectors(
        potential_variables.f_matrix[atomindex, :],
        potential_variables.new_f_vec,
        atomindex
    )

    potential_variables.new_g_matrix = copy(potential_variables.g_matrix)

    potential_variables.new_g_matrix = total_symm_neigh!(
        potential_variables.new_g_matrix,
        config.pos,
        trial_pos,
        dist2_mat,
        new_dist2_vec,
        potential_variables.f_matrix,
        potential_variables.new_f_vec,
        atomindex,
        neighbour_indices_from_f,
        pot.radsymfunctions,
        pot.angsymfunctions,
        Nrad,
        Nang
    )

    return potential_variables
end

function energy_update_neigh!(
    ensemblevariables,
    config,
    potential_variables,
    dist2_mat,
    new_dist2_vec,
    en_tot,
    pot
)
    if any(new_dist2_vec[i] < pot.boundary for i in eachindex(new_dist2_vec) if i != ensemblevariables.index)
        new_en = 100.0
    else
        potential_variables = get_new_state_vars_neigh!(
            ensemblevariables.trial_move,
            ensemblevariables.index,
            config,
            potential_variables,
            dist2_mat,
            new_dist2_vec,
            pot
        )

        potential_variables, new_en = calc_new_runner_energy!(potential_variables, pot)
    end

    return potential_variables, new_en
end
=#
# ------------------------------------------------------------
# Correctness comparison
# ------------------------------------------------------------

pv_old = deepcopy(mc_state.potential_variables)
pv_neigh = deepcopy(mc_state.potential_variables)

pv_old, en_old = energy_update!(
    mc_state.ensemble_variables,
    mc_state.config,
    pv_old,
    mc_state.dist2_mat,
    new_dist2_vec,
    mc_state.en_tot,
    pot
)

pv_neigh, en_neigh = energy_update_neigh!(
    mc_state.ensemble_variables,
    mc_state.config,
    pv_neigh,
    mc_state.dist2_mat,
    new_dist2_vec,
    mc_state.en_tot,
    pot
)

println()
println("Correctness test")
println("energy old method = ", en_old)
println("energy neigh = ", en_neigh)
println("energy difference = ", abs(en_old - en_neigh))
println("G difference = ", maximum(abs.(pv_old.new_g_matrix .- pv_neigh.new_g_matrix)))
println("new atom-energy vector difference = ", maximum(abs.(pv_old.new_en_atom .- pv_neigh.new_en_atom)))

# ------------------------------------------------------------
# Benchmark comparison
# ------------------------------------------------------------

println()
println("Benchmarking old energy_update! path")
@btime energy_update!(
    $(mc_state.ensemble_variables),
    $(mc_state.config),
    deepcopy($(mc_state.potential_variables)),
    $(mc_state.dist2_mat),
    $new_dist2_vec,
    $(mc_state.en_tot),
    $pot
);

println()
println("Benchmarking neighbour-restricted energy_update_neigh! path")
@btime energy_update_neigh!(
    $(mc_state.ensemble_variables),
    $(mc_state.config),
    deepcopy($(mc_state.potential_variables)),
    $(mc_state.dist2_mat),
    $new_dist2_vec,
    $(mc_state.en_tot),
    $pot
);

  Activating project at `~/ParallelTemperingMonteCarlo.jl`


Generated Cu147
N = 147
central NN distance = 4.575916480942233

Correctness test
energy old = -22.428406923912934
energy neigh = -22.428406923912934
energy difference = 0.0
G difference = 0.0
new atom-energy vector difference = 0.0

Benchmarking old energy_update! path
  49.600 ms (117 allocations: 587.38 KiB)

Benchmarking neighbour-restricted energy_update_neigh! path
  6.006 ms (110 allocations: 589.16 KiB)


In [ ]:
using BenchmarkTools
using Statistics
using LinearAlgebra
using StaticArrays

shells = [3, 4, 5]   # 147, 309, 561
target_nn_bohr = 4.575916480942233
displacement = SVector{3,Float64}(0.05, 0.0, 0.0)

n_timing_repeats = 3

results = []

for shell in shells
    positions_raw = deduplicate_positions(mackay_icosahedron(shell))
    raw_nn = central_neighbour_distance_scale(positions_raw)
    positions = [(target_nn_bohr / raw_nn) * p for p in positions_raw]

    N = length(positions)
    cluster_radius = maximum(norm(p) for p in positions)
    bc = SphericalBC(radius = cluster_radius + 5.0)
    config = Config(positions, bc)

    ensemble = NVT(N)
    temp = 1000.0
    beta = 1.0 / temp

    println()
    println("Initialising MCState for N = ", N)
    mc_state = MCState(temp, beta, config, ensemble, pot)

    atomindex = find_central_atom(config.pos)
    trial_move = config.pos[atomindex] + displacement

    new_dist2_vec = [
        sum((trial_move .- config.pos[j]).^2)
        for j in eachindex(config.pos)
    ]
    new_dist2_vec[atomindex] = 0.0

    mc_state.ensemble_variables.index = atomindex
    mc_state.ensemble_variables.trial_move = trial_move

    new_f_vec = cutoff_function.(sqrt.(new_dist2_vec), Ref(pot.r_cut))

    neighbour_indices_from_f = neighbour_union_from_cutoff_vectors(
        mc_state.potential_variables.f_matrix[atomindex, :],
        new_f_vec,
        atomindex
    )

    neighbour_fraction = length(neighbour_indices_from_f) / (N - 1)

    # correctness check, also serves as warmup
    pv_old = deepcopy(mc_state.potential_variables)
    pv_neigh = deepcopy(mc_state.potential_variables)

    pv_old, en_old = energy_update!(
        mc_state.ensemble_variables,
        mc_state.config,
        pv_old,
        mc_state.dist2_mat,
        new_dist2_vec,
        mc_state.en_tot,
        pot
    )

    pv_neigh, en_neigh = energy_update_neigh!(
        mc_state.ensemble_variables,
        mc_state.config,
        pv_neigh,
        mc_state.dist2_mat,
        new_dist2_vec,
        mc_state.en_tot,
        pot
    )

    energy_diff = abs(en_old - en_neigh)
    G_diff = maximum(abs.(pv_old.new_g_matrix .- pv_neigh.new_g_matrix))

    println("N = ", N)
    println("central atomindex = ", atomindex)
    println("neighbour count = ", length(neighbour_indices_from_f))
    println("neighbour fraction = ", neighbour_fraction)
    println("energy difference = ", energy_diff)
    println("G difference = ", G_diff)

    #runtime comparison for median runtime between neigbhour and regular energy update.
    old_times = Float64[]
    neigh_times = Float64[]

    for r in 1:n_timing_repeats
        GC.gc() #garbage collectior

        t_old = @elapsed energy_update!(
            mc_state.ensemble_variables,
            mc_state.config,
            deepcopy(mc_state.potential_variables),
            mc_state.dist2_mat,
            new_dist2_vec,
            mc_state.en_tot,
            pot
        )

        GC.gc() #garbage collectior

        t_neigh = @elapsed energy_update_neigh!(
            mc_state.ensemble_variables,
            mc_state.config,
            deepcopy(mc_state.potential_variables),
            mc_state.dist2_mat,
            new_dist2_vec,
            mc_state.en_tot,
            pot
        )

        push!(old_times, t_old)
        push!(neigh_times, t_neigh)
    end

    old_time = median(old_times)
    neigh_time = median(neigh_times)
    speedup = old_time / neigh_time

    println("old median time = ", old_time, " s")
    println("neigh median time = ", neigh_time, " s")
    println("speedup = ", speedup)

    push!(results, (
        N = N,
        neighbour_count = length(neighbour_indices_from_f),
        neighbour_fraction = neighbour_fraction,
        energy_diff = energy_diff,
        G_diff = G_diff,
        old_time = old_time,
        neigh_time = neigh_time,
        speedup = speedup
    ))
end

println()
println("Scaling summary relative to N = ", results[1].N)

base_old = results[1].old_time
base_neigh = results[1].neigh_time

for r in results
    println()
    println("N = ", r.N)
    println("neighbour count = ", r.neighbour_count)
    println("neighbour fraction = ", r.neighbour_fraction)
    println("old / old_147 = ", r.old_time / base_old)
    println("neigh / neigh_147 = ", r.neigh_time / base_neigh)
    println("speedup old/neigh = ", r.speedup)
    println("energy diff = ", r.energy_diff)
    println("G diff = ", r.G_diff)
end


Initialising MCState for N = 147
N = 147
central atomindex = 33
neighbour count = 74
neighbour fraction = 0.5068493150684932
energy difference = 0.0
G difference = 0.0
old median time = 0.065717422 s
neigh median time = 0.011971856 s
speedup = 5.489326132890339

Initialising MCState for N = 309
N = 309
central atomindex = 46
neighbour count = 74
neighbour fraction = 0.24025974025974026
energy difference = 0.0
G difference = 0.0
old median time = 0.375992782 s
neigh median time = 0.019387312 s
speedup = 19.39375515285461

Initialising MCState for N = 561
N = 561
central atomindex = 79
neighbour count = 74
neighbour fraction = 0.13214285714285715
energy difference = 0.0
G difference = 0.0
old median time = 1.465664186 s
neigh median time = 0.024313569 s
speedup = 60.281737576248055

Scaling summary relative to N = 147

N = 147
neighbour count = 74
neighbour fraction = 0.5068493150684932
old / old_147 = 1.0
neigh / neigh_147 = 1.0
speedup old/neigh = 5.489326132890339
energy diff = 0.0
G

This cell compiles all functions needed to run later code cells.

In [3]:
using Pkg
Pkg.activate("/home/rev/ParallelTemperingMonteCarlo.jl")

using ParallelTemperingMonteCarlo

using StaticArrays
using LinearAlgebra
using DelimitedFiles
using BenchmarkTools
using Statistics
using Printf

function mackay_icosahedron(shell::Int; target_nn_bohr=nothing)
    φ = (1 + sqrt(5.0)) / 2

    verts = SVector{3,Float64}[]

    for s1 in (-1.0, 1.0), s2 in (-1.0, 1.0)
        push!(verts, SVector(0.0, s1, s2*φ))
        push!(verts, SVector(s1, s2*φ, 0.0))
        push!(verts, SVector(s1*φ, 0.0, s2))
    end

    # Find triangular faces of the icosahedron
    edge = minimum(norm(verts[i] - verts[j]) for i in eachindex(verts), j in eachindex(verts) if i < j)

    faces = Tuple{Int,Int,Int}[]

    for i in 1:length(verts)-2
        for j in i+1:length(verts)-1
            for k in j+1:length(verts)
                if abs(norm(verts[i] - verts[j]) - edge) < 1e-8 &&
                   abs(norm(verts[i] - verts[k]) - edge) < 1e-8 &&
                   abs(norm(verts[j] - verts[k]) - edge) < 1e-8
                    push!(faces, (i, j, k))
                end
            end
        end
    end

    # Fill each tetrahedron: centre + one triangular face
    pointset = Set{NTuple{3,Float64}}()

    for (a, b, c) in faces
        v1, v2, v3 = verts[a], verts[b], verts[c]

        for i in 0:shell
            for j in 0:(shell - i)
                for k in 0:(shell - i - j)
                    p = (i*v1 + j*v2 + k*v3) / shell
                    key = (
                        round(p[1], digits=10),
                        round(p[2], digits=10),
                        round(p[3], digits=10)
                    )
                    push!(pointset, key)
                end
            end
        end
    end

    positions = [SVector{3,Float64}(p) for p in pointset]

    # Recentre at origin
    centre = sum(positions) / length(positions)
    positions = [p - centre for p in positions]

    # Optional scaling
    if target_nn_bohr !== nothing
        min_dist = minimum(
            norm(positions[i] - positions[j])
            for i in eachindex(positions), j in eachindex(positions) if i < j
        )

        scale = target_nn_bohr / min_dist
        positions = [scale * p for p in positions]
    end

    return positions
end

function scale_by_central_neighbours(positions; target_nn_bohr)
    centre_index = find_central_atom(positions)
    x0 = positions[centre_index]

    dists = sort([
        norm(positions[i] - x0)
        for i in eachindex(positions)
        if i != centre_index
    ])

    current_nn = mean(dists[1:12])
    scale = target_nn_bohr / current_nn

    return [scale * p for p in positions]
end

function deduplicate_positions(positions; tol=1e-8)
    seen = Set{NTuple{3,Int}}()
    unique_positions = SVector{3,Float64}[]

    for p in positions
        key = (
            round(Int, p[1] / tol),
            round(Int, p[2] / tol),
            round(Int, p[3] / tol)
        )

        if !(key in seen)
            push!(seen, key)
            push!(unique_positions, p)
        end
    end

    return unique_positions
end

function find_central_atom(positions)
    return argmin([norm(p) for p in positions])
end

function central_neighbour_distance_scale(positions; n_neigh=12, tol=1e-8)
    atomindex = find_central_atom(positions)
    x0 = positions[atomindex]

    dists = sort([
        norm(positions[i] - x0)
        for i in eachindex(positions)
        if i != atomindex && norm(positions[i] - x0) > tol
    ])

    return mean(dists[1:n_neigh])
end

# ------------------------------------------------------------
# Build RuNNer potential
# ------------------------------------------------------------

data_path = "/home/rev/ParallelTemperingMonteCarlo.jl/scripts/data"

X = [ 11 0.001 0.000 11.338
      10 0.001 0.000 11.338
      11 0.020 0.000 11.338
      10 0.020 0.000 11.338
      11 0.035 0.000 11.338
      10 0.035 0.000 11.338
      11 0.100 0.000 11.338
      10 0.100 0.000 11.338
      11 0.400 0.000 11.338
      10 0.400 0.000 11.338 ]

V = [[0.0001,1,1,11.338],[0.0001,-1,2,11.338],
     [0.003,-1,1,11.338],[0.003,-1,2,11.338],
     [0.008,-1,1,11.338],[0.008,-1,2,11.338],[0.008,1,2,11.338],
     [0.015,1,1,11.338],[0.015,-1,2,11.338],[0.015,-1,4,11.338],
     [0.015,-1,16,11.338],[0.025,-1,1,11.338],[0.025,1,1,11.338],
     [0.025,1,2,11.338],[0.025,-1,4,11.338],[0.025,-1,16,11.338],
     [0.025,1,16,11.338],[0.045,1,1,11.338],[0.045,-1,2,11.338],
     [0.045,-1,4,11.338],[0.045,1,4,11.338],[0.045,1,16,11.338],
     [0.08,1,1,11.338],[0.08,-1,2,11.338],
     [0.08,-1,4,11.338],[0.08,1,4,11.338]]

T = [111, 110, 100]

scalingvalues = readdlm(joinpath(data_path, "scaling.data"))
G_value_vec = []

for row in eachrow(scalingvalues[1:88, :])
    push!(G_value_vec, [row[4], row[3]])
end

radsymmvec = RadialType2{Float64}[]
angularsymmvec = AngularType3{Float64}[]

for symmindex in eachindex(eachrow(X))
    row = X[symmindex, :]
    push!(radsymmvec, RadialType2{Float64}(row[2], row[4], Int(row[1]), G_value_vec[symmindex]))
end

let n_index = 10
    for element in V
        for types in T
            n_index += 1
            push!(angularsymmvec,
                AngularType3{Float64}(element[1], element[2], element[3], 11.338, types, G_value_vec[n_index])
            )
        end
    end
end

num_nodes = Int32[88, 20, 20, 1]
activation_functions = Int32[1, 2, 2, 1]
weights = vec(readdlm(joinpath(data_path, "weights.029.data")))

nnp = NeuralNetworkPotential(num_nodes, activation_functions, weights)
pot = RuNNerPotential(nnp, radsymmvec, angularsymmvec)

function neighbour_count_from_f_matrix(f_matrix, atomindex)
    k = 0

    for j in axes(f_matrix, 2)
        if j != atomindex && f_matrix[atomindex, j] != 0.0
            k += 1
        end
    end

    return k
end


  Activating project at `~/ParallelTemperingMonteCarlo.jl`


neighbour_count_from_f_matrix (generic function with 1 method)

Compares one MC move for correctness and timing for each shell size for the old-method vs neighbvour-restructed version.

In [ ]:
using BenchmarkTools
using Statistics
using LinearAlgebra
using StaticArrays
using Printf

shells = [2, 3, 4, 5, 6]   # 55, 147, 309, 561, 923
target_nn_bohr = 4.575916480942233
displacement = SVector{3,Float64}(0.05, 0.0, 0.0)

n_timing_repeats = 3

table_results = []

function neighbour_count_from_f_matrix(f_matrix, atomindex)
    k = 0

    for j in axes(f_matrix, 2)
        if j != atomindex && f_matrix[atomindex, j] != 0.0
            k += 1
        end
    end

    return k
end

for shell in shells
    positions_raw = deduplicate_positions(mackay_icosahedron(shell))
    raw_nn = central_neighbour_distance_scale(positions_raw)
    positions = [(target_nn_bohr / raw_nn) * p for p in positions_raw]

    N = length(positions)
    cluster_radius = maximum(norm(p) for p in positions)
    bc = SphericalBC(radius = cluster_radius + 5.0)
    config = Config(positions, bc)

    ensemble = NVT(N)
    temp = 1000.0
    beta = 1.0 / temp

    println()
    println("Initialising MCState for N = ", N)
    mc_state = MCState(temp, beta, config, ensemble, pot)

    atomindex = find_central_atom(config.pos)
    trial_move = config.pos[atomindex] + displacement

    new_dist2_vec = [
        sum((trial_move .- config.pos[j]).^2)
        for j in eachindex(config.pos)
    ]
    new_dist2_vec[atomindex] = 0.0

    mc_state.ensemble_variables.index = atomindex
    mc_state.ensemble_variables.trial_move = trial_move

    new_f_vec = cutoff_function.(sqrt.(new_dist2_vec), Ref(pot.r_cut))

    neighbour_indices_from_f = neighbour_union_from_cutoff_vectors(
        mc_state.potential_variables.f_matrix[atomindex, :],
        new_f_vec,
        atomindex
    )

    central_k = neighbour_count_from_f_matrix(
        mc_state.potential_variables.f_matrix,
        atomindex
    )

    all_ks = [
        neighbour_count_from_f_matrix(mc_state.potential_variables.f_matrix, i)
        for i in 1:N
    ]

    median_k = median(all_ks)

    # Correctness check, also serves as warmup
    pv_old = deepcopy(mc_state.potential_variables)
    pv_neigh = deepcopy(mc_state.potential_variables)

    pv_old, en_old = energy_update!(
        mc_state.ensemble_variables,
        mc_state.config,
        pv_old,
        mc_state.dist2_mat,
        new_dist2_vec,
        mc_state.en_tot,
        pot
    )

    pv_neigh, en_neigh = energy_update_neigh!(
        mc_state.ensemble_variables,
        mc_state.config,
        pv_neigh,
        mc_state.dist2_mat,
        new_dist2_vec,
        mc_state.en_tot,
        pot
    )

    energy_diff = abs(en_old - en_neigh)
    G_diff = maximum(abs.(pv_old.new_g_matrix .- pv_neigh.new_g_matrix))

    move_energy_change = en_old - mc_state.en_tot

    @printf("energy old method             = %.17e\n", en_old)
    @printf("energy neigh method           = %.17e\n", en_neigh)
    @printf("energy difference       = %.17e\n", energy_diff)
    @printf("move energy change ΔE   = %.17e\n", move_energy_change)
    @printf("G difference            = %.17e\n", G_diff)

    old_times = Float64[]
    neigh_times = Float64[]

    for r in 1:n_timing_repeats
        GC.gc() #garbage collectior

        t_old = @elapsed energy_update!(
            mc_state.ensemble_variables,
            mc_state.config,
            deepcopy(mc_state.potential_variables),
            mc_state.dist2_mat,
            new_dist2_vec,
            mc_state.en_tot,
            pot
        )

        GC.gc() #garbage collectior

        t_neigh = @elapsed energy_update_neigh!(
            mc_state.ensemble_variables,
            mc_state.config,
            deepcopy(mc_state.potential_variables),
            mc_state.dist2_mat,
            new_dist2_vec,
            mc_state.en_tot,
            pot
        )

        push!(old_times, t_old)
        push!(neigh_times, t_neigh)
    end

    old_time = median(old_times)
    neigh_time = median(neigh_times)
    speedup = old_time / neigh_time

    push!(table_results, (
        shell = shell,
        N = N,
        initial_energy = mc_state.en_tot,
        central_atom = atomindex,
        central_k = central_k,
        central_fraction = central_k / (N - 1),
        union_k = length(neighbour_indices_from_f),
        union_fraction = length(neighbour_indices_from_f) / (N - 1),
        median_k = median_k,
        median_fraction = median_k / (N - 1),
        old_time = old_time,
        neigh_time = neigh_time,
        speedup = speedup,
        move_energy_change,
        energy_diff = energy_diff,
        G_diff = G_diff
    ))

    println("N = ", N)
    println("initial energy = ", mc_state.en_tot)
    println("central k = ", central_k)
    println("union k after move for central k = ", length(neighbour_indices_from_f))
    println("median k for cluster = ", median_k)
    println("old time for central = ", old_time)
    println("neigh time for central = ", neigh_time)
    println("speedup = ", speedup)
    println("energy diff = ", energy_diff)
    println("G diff = ", G_diff)
end


println()
println("----------------------------------------------------------------------------------------------------------------")
println(rpad("N",8),
        rpad("ΔE move",18),
        rpad("|ΔE method|",18),
        rpad("Central k",12),
        rpad("Median k",12),
        rpad("Old (s)",12),
        rpad("Neighbour (s)",15),
        rpad("Speedup",10))
println("----------------------------------------------------------------------------------------------------------------")

for r in table_results
    @printf("%-8d %-18.8e %-24.17e %-12d %-12.1f %-12.5f %-15.5f %-10.2f\n",
    r.N,
    r.move_energy_change,
    r.energy_diff,
    r.central_k,
    r.median_k,
    r.old_time,
    r.neigh_time,
    r.speedup)
end

println("----------------------------------------------------------------------------------------------------------------")


Initialising MCState for N = 55
energy old method             = -7.69234604379549580e+00
energy neigh method           = -7.69234604379549580e+00
energy difference       = 0.00000000000000000e+00
move energy change ΔE   = 1.00066920974306584e-04
G difference            = 0.00000000000000000e+00
N = 55
initial energy = -7.69244611071647
central k = 54
union k after move for central k = 54
median k for cluster = 31.0
old time for central = 0.00805698
neigh time for central = 0.00678231
speedup = 1.1879403919903397
energy diff = 0.0
G diff = 0.0

Initialising MCState for N = 147
energy old method             = -2.24284069239129344e+01
energy neigh method           = -2.24284069239129344e+01
energy difference       = 0.00000000000000000e+00
move energy change ΔE   = 8.57373318119414307e-05
G difference            = 0.00000000000000000e+00
N = 147
initial energy = -22.428492661244746
central k = 74
union k after move for central k = 74
median k for cluster = 43.0
old time for central = 0.1

In [5]:
function get_energy_neigh!(mc_state, pot, movetype::String)
    if movetype == "atommove"
        mc_state.potential_variables, mc_state.new_en = energy_update_neigh!(
            mc_state.ensemble_variables,
            mc_state.config,
            mc_state.potential_variables,
            mc_state.dist2_mat,
            mc_state.new_dist2_vec,
            mc_state.en_tot,
            pot
        )
    else
        error("get_energy_neigh! currently only supports atommove")
    end

    return mc_state
end

function mc_move_neigh!(mc_state, move_strat, pot, ensemble)
    N = length(mc_state.config.pos)

    mc_state.ensemble_variables.index = rand(1:N)

    movetype = move_strat.movestrat[mc_state.ensemble_variables.index]

    mc_state = generate_move!(mc_state, movetype)
    mc_state = get_energy_neigh!(mc_state, pot, movetype)

    acc_test!(mc_state, ensemble, movetype)

    return mc_state
end

function mc_step_neigh!(mc_states, move_strat, pot, ensemble, n_steps::Int)
    Threads.@threads for s in eachindex(mc_states)
        state = mc_states[s]

        for i_step in 1:n_steps
            state = mc_move_neigh!(state, move_strat, pot, ensemble)
        end

        mc_states[s] = state
    end

    return mc_states
end

function acc_test_with_rand!(mc_state::MCState, ensemble::Etype, movetype::String, r::Float64) where Etype
    if metropolis_condition(movetype, mc_state, ensemble) >= r
        swap_config!(mc_state, movetype)
    end

    return mc_state
end

acc_test_with_rand! (generic function with 1 method)

The below tests  the cu55 cluster via old method vs the neighbour restricted method for correctness and speed for one full MC cycle of N atom moves.

In [4]:
using Random
using Printf
using Statistics
using ParallelTemperingMonteCarlo


function get_energy_neigh!(mc_state, pot, movetype::String)
    if movetype == "atommove"
        mc_state.potential_variables, mc_state.new_en = energy_update_neigh!(
            mc_state.ensemble_variables,
            mc_state.config,
            mc_state.potential_variables,
            mc_state.dist2_mat,
            mc_state.new_dist2_vec,
            mc_state.en_tot,
            pot
        )
    else
        error("get_energy_neigh! currently only supports atommove")
    end

    return mc_state
end

function acc_test_with_rand!(mc_state::MCState, ensemble, movetype::String, r_accept::Float64)
    if metropolis_condition(movetype, mc_state, ensemble) >= r_accept
        swap_config!(mc_state, movetype)
    end

    return mc_state
end

function mc_move_neigh!(mc_state, move_strat, pot, ensemble)
    N = length(mc_state.config.pos)

    mc_state.ensemble_variables.index = rand(1:N)
    movetype = move_strat.movestrat[mc_state.ensemble_variables.index]

    mc_state = generate_move!(mc_state, movetype)
    mc_state = get_energy_neigh!(mc_state, pot, movetype)
    acc_test!(mc_state, ensemble, movetype)

    return mc_state
end

function mc_step_neigh_single!(
    mc_state,
    move_strat,
    pot,
    ensemble,
    n_steps::Int,
)
    for _ in 1:n_steps
        mc_state = mc_move_neigh!(
            mc_state,
            move_strat,
            pot,
            ensemble,
        )
    end

    return mc_state
end

function mc_step_old_single!(mc_state, move_strat, pot, ensemble, n_steps::Int)
    for i_step in 1:n_steps
        mc_state = mc_move!(mc_state, move_strat, pot, ensemble)
    end

    return mc_state
end

function paired_mc_step_compare!(mc_state_old, mc_state_neigh, move_strat, pot, ensemble, n_steps::Int)
    trial_energy_diffs = Float64[]
    trial_g_diffs = Float64[]
    accepted_flags = Bool[]

    for step in 1:n_steps
        N = length(mc_state_old.config.pos)

        # Same atom selected for both methods
        atomindex = rand(1:N)
        movetype = move_strat.movestrat[atomindex]

        mc_state_old.ensemble_variables.index = atomindex
        mc_state_neigh.ensemble_variables.index = atomindex

        # Generate trial move using old state
        mc_state_old = generate_move!(mc_state_old, movetype)

        # Copy the exact same proposed move into neighbour state
        mc_state_neigh.ensemble_variables.trial_move = mc_state_old.ensemble_variables.trial_move
        mc_state_neigh.new_dist2_vec = copy(mc_state_old.new_dist2_vec)

        # Compute trial energies
        mc_state_old = get_energy!(mc_state_old, pot, movetype)
        mc_state_neigh = get_energy_neigh!(mc_state_neigh, pot, movetype)

        push!(trial_energy_diffs, abs(mc_state_old.new_en - mc_state_neigh.new_en))
        push!(trial_g_diffs, maximum(abs.(
            mc_state_old.potential_variables.new_g_matrix .-
            mc_state_neigh.potential_variables.new_g_matrix
        )))

        # Same accept/reject random number
        r_accept = rand()

        old_energy_before = mc_state_old.en_tot

        acc_test_with_rand!(mc_state_old, ensemble, movetype, r_accept)
        acc_test_with_rand!(mc_state_neigh, ensemble, movetype, r_accept)

        push!(accepted_flags, mc_state_old.en_tot != old_energy_before)
    end

    return (
        mc_state_old = mc_state_old,
        mc_state_neigh = mc_state_neigh,
        max_trial_energy_diff = maximum(trial_energy_diffs),
        max_trial_g_diff = maximum(trial_g_diffs),
        final_energy_diff = abs(mc_state_old.en_tot - mc_state_neigh.en_tot),
        final_g_diff = maximum(abs.(
            mc_state_old.potential_variables.g_matrix .-
            mc_state_neigh.potential_variables.g_matrix
        )),
        n_accepted = count(accepted_flags)
    )
end

# ------------------------------------------------------------
# Build Cu55 test state
# ------------------------------------------------------------

shell = 2
target_nn_bohr = 4.575916480942233

positions_raw = deduplicate_positions(mackay_icosahedron(shell))
raw_nn = central_neighbour_distance_scale(positions_raw)
positions = [(target_nn_bohr / raw_nn) * p for p in positions_raw]

N = length(positions)
cluster_radius = maximum(norm(p) for p in positions)
bc = SphericalBC(radius = cluster_radius + 5.0)
config = Config(positions, bc)

ensemble = NVT(N)
move_strat = MoveStrategy(ensemble)

temp = 1000.0
beta = 1.0 / temp

mc_state = MCState(temp, beta, config, ensemble, pot)

# ------------------------------------------------------------
# Paired correctness test: one full MC cycle = N attempted moves
# ------------------------------------------------------------

Random.seed!(1234)

mc_state_old = deepcopy(mc_state)
mc_state_neigh = deepcopy(mc_state)

result = paired_mc_step_compare!(
    mc_state_old,
    mc_state_neigh,
    move_strat,
    pot,
    ensemble,
    N
)

println("Cu55 paired one-cycle correctness test")
println("N attempted moves = ", N)
println("accepted moves = ", result.n_accepted)
@printf("max trial energy diff = %.17e\n", result.max_trial_energy_diff)
@printf("max trial G diff      = %.17e\n", result.max_trial_g_diff)
@printf("final energy diff     = %.17e\n", result.final_energy_diff)
@printf("final G diff          = %.17e\n", result.final_g_diff)

# ------------------------------------------------------------
# Timing test: old vs neighbour one-cycle runtime
# ------------------------------------------------------------

n_timing_repeats = 5

old_times = Float64[]
neigh_times = Float64[]

for r in 1:n_timing_repeats
    seed = 1234 + r
    Random.seed!(seed)
    t_old = @elapsed mc_step_old_single!(
        deepcopy(mc_state),
        move_strat,
        pot,
        ensemble,
        N
    )

    Random.seed!(seed)
    t_neigh = @elapsed mc_step_neigh_single!(
        deepcopy(mc_state),
        move_strat,
        pot,
        ensemble,
        N
    )

    push!(old_times, t_old)
    push!(neigh_times, t_neigh)
end

old_time = median(old_times)
neigh_time = median(neigh_times)

println()
print("attempted moves = ", N)
println("timing repeats = ", n_timing_repeats)
println("Cu55 one-cycle timing")
println("old_times = ", old_times)
println("neigh_times = ", neigh_times)
println("old median time = ", old_time, " s")
println("neigh median time = ", neigh_time, " s")
println("speedup = ", old_time / neigh_time)

Cu55 paired one-cycle correctness test
N attempted moves = 55
accepted moves = 55
max trial energy diff = 0.00000000000000000e+00
max trial G diff      = 0.00000000000000000e+00
final energy diff     = 0.00000000000000000e+00
final G diff          = 0.00000000000000000e+00

attempted moves = 55timing repeats = 5
Cu55 one-cycle timing
old_times = [0.840132444, 0.577060308, 0.460295547, 0.495220499, 0.448212249]
neigh_times = [0.204446391, 0.122500094, 0.146053962, 0.16411579, 0.164643693]
old median time = 0.495220499 s
neigh median time = 0.16411579 s
speedup = 3.0175067188842704


The below tests any shell's old method vs the neighbour restricted method for correctness and speed for one full MC cycle of N atom moves.

In [ ]:
using Random
using Printf
using Statistics

function get_energy_neigh!(mc_state, pot, movetype::String)
    if movetype == "atommove"
        mc_state.potential_variables, mc_state.new_en = energy_update_neigh!(
            mc_state.ensemble_variables,
            mc_state.config,
            mc_state.potential_variables,
            mc_state.dist2_mat,
            mc_state.new_dist2_vec,
            mc_state.en_tot,
            pot
        )
    else
        error("get_energy_neigh! currently only supports atommove")
    end

    return mc_state
end

function acc_test_with_rand!(mc_state::MCState, ensemble, movetype::String, r_accept::Float64)
    if metropolis_condition(movetype, mc_state, ensemble) >= r_accept
        swap_config!(mc_state, movetype)
    end

    return mc_state
end

function mc_move_neigh!(mc_state, move_strat, pot, ensemble)
    N = length(mc_state.config.pos)

    mc_state.ensemble_variables.index = rand(1:N)
    movetype = move_strat.movestrat[mc_state.ensemble_variables.index]

    mc_state = generate_move!(mc_state, movetype)
    mc_state = get_energy_neigh!(mc_state, pot, movetype)
    acc_test!(mc_state, ensemble, movetype)

    return mc_state
end

function mc_step_neigh_single!(
    mc_state,
    move_strat,
    pot,
    ensemble,
    n_steps::Int,
)
    for _ in 1:n_steps
        mc_state = mc_move_neigh!(
            mc_state,
            move_strat,
            pot,
            ensemble,
        )
    end

    return mc_state
end

function mc_step_old_single!(mc_state, move_strat, pot, ensemble, n_steps::Int)
    for i_step in 1:n_steps
        mc_state = mc_move!(mc_state, move_strat, pot, ensemble)
    end

    return mc_state
end

function paired_mc_step_compare!(mc_state_old, mc_state_neigh, move_strat, pot, ensemble, n_steps::Int)
    trial_energy_diffs = Float64[]
    trial_g_diffs = Float64[]
    accepted_flags = Bool[]

    for step in 1:n_steps
        N = length(mc_state_old.config.pos)

        # Same atom selected for both methods
        atomindex = rand(1:N)
        movetype = move_strat.movestrat[atomindex]

        mc_state_old.ensemble_variables.index = atomindex
        mc_state_neigh.ensemble_variables.index = atomindex

        # Generate trial move using old state
        mc_state_old = generate_move!(mc_state_old, movetype)

        # Copy the exact same proposed move into neighbour state
        mc_state_neigh.ensemble_variables.trial_move = mc_state_old.ensemble_variables.trial_move
        mc_state_neigh.new_dist2_vec = copy(mc_state_old.new_dist2_vec)

        # Compute trial energies
        mc_state_old = get_energy!(mc_state_old, pot, movetype)
        mc_state_neigh = get_energy_neigh!(mc_state_neigh, pot, movetype)

        push!(trial_energy_diffs, abs(mc_state_old.new_en - mc_state_neigh.new_en))
        push!(trial_g_diffs, maximum(abs.(
            mc_state_old.potential_variables.new_g_matrix .-
            mc_state_neigh.potential_variables.new_g_matrix
        )))

        # Same accept/reject random number
        r_accept = rand()

        old_energy_before = mc_state_old.en_tot

        acc_test_with_rand!(mc_state_old, ensemble, movetype, r_accept)
        acc_test_with_rand!(mc_state_neigh, ensemble, movetype, r_accept)

        push!(accepted_flags, mc_state_old.en_tot != old_energy_before)
    end

    return (
        mc_state_old = mc_state_old,
        mc_state_neigh = mc_state_neigh,
        max_trial_energy_diff = maximum(trial_energy_diffs),
        max_trial_g_diff = maximum(trial_g_diffs),
        final_energy_diff = abs(mc_state_old.en_tot - mc_state_neigh.en_tot),
        final_g_diff = maximum(abs.(
            mc_state_old.potential_variables.g_matrix .-
            mc_state_neigh.potential_variables.g_matrix
        )),
        n_accepted = count(accepted_flags)
    )
end

# ------------------------------------------------------------
# Build Cluster test state
# ------------------------------------------------------------

shell = 5
target_nn_bohr = 4.575916480942233

positions_raw = deduplicate_positions(mackay_icosahedron(shell))
raw_nn = central_neighbour_distance_scale(positions_raw)
positions = [(target_nn_bohr / raw_nn) * p for p in positions_raw]

N = length(positions)
cluster_radius = maximum(norm(p) for p in positions)
bc = SphericalBC(radius = cluster_radius + 5.0)
config = Config(positions, bc)

ensemble = NVT(N)
move_strat = MoveStrategy(ensemble)

temp = 1000.0
beta = 1.0 / temp

mc_state = MCState(temp, beta, config, ensemble, pot)

# ------------------------------------------------------------
# Paired correctness test: one full MC cycle = N attempted moves
# ------------------------------------------------------------

Random.seed!(1234)

mc_state_old = deepcopy(mc_state)
mc_state_neigh = deepcopy(mc_state)

result = paired_mc_step_compare!(
    mc_state_old,
    mc_state_neigh,
    move_strat,
    pot,
    ensemble,
    N
)

println("Cu561 paired one-cycle correctness test")
println("N attempted moves = ", N)
println("accepted moves = ", result.n_accepted)
@printf("max trial energy diff = %.17e\n", result.max_trial_energy_diff)
@printf("max trial G diff      = %.17e\n", result.max_trial_g_diff)
@printf("final energy diff     = %.17e\n", result.final_energy_diff)
@printf("final G diff          = %.17e\n", result.final_g_diff)

# ------------------------------------------------------------
# Timing test: old vs neighbour one-cycle runtime
# ------------------------------------------------------------

n_timing_repeats = 5

old_times = Float64[]
neigh_times = Float64[]

for r in 1:n_timing_repeats
    seed = 1234 + r
    Random.seed!(seed)
    t_old = @elapsed mc_step_old_single!(
        deepcopy(mc_state),
        move_strat,
        pot,
        ensemble,
        N
    )

    Random.seed!(seed)
    t_neigh = @elapsed mc_step_neigh_single!(
        deepcopy(mc_state),
        move_strat,
        pot,
        ensemble,
        N
    )

    push!(old_times, t_old)
    push!(neigh_times, t_neigh)
end

old_time = median(old_times)
neigh_time = median(neigh_times)

println()
println("attempted moves = ", N)
println("timing repeats = ", n_timing_repeats)
println("Cu55 one-cycle timing")
println("old_times = ", old_times)
println("neigh_times = ", neigh_times)
println("old median time = ", old_time, " s")
println("neigh median time = ", neigh_time, " s")
println("speedup = ", old_time / neigh_time)

Cu561 paired one-cycle correctness test
N attempted moves = 561
accepted moves = 561
max trial energy diff = 0.00000000000000000e+00
max trial G diff      = 0.00000000000000000e+00
final energy diff     = 0.00000000000000000e+00
final G diff          = 0.00000000000000000e+00

attempted moves = 561
timing repeats = 5
Cu55 one-cycle timing
old_times = [586.959595914, 562.205805849, 534.685667857, 555.299374189, 609.601799723]
neigh_times = [5.604466549, 5.443323633, 5.192182683, 13.79524548, 5.46527978]
old median time = 562.205805849 s
neigh median time = 5.46527978 s
speedup = 102.86862310441497


The below cell is for running only the neighbour-restricted version of an MC step as to run larger shells that would take too long to compare against the old method. Simply change the shell number to produce the results. In particular, note that the neighbour-restricted 923 shell runs faster than the old-method 147 cluster, showing that we can now partically run this large cluster.

In [ ]:
using Random
using Printf
using Statistics

function get_energy_neigh!(mc_state, pot, movetype::String)
    if movetype == "atommove"
        mc_state.potential_variables, mc_state.new_en = energy_update_neigh!(
            mc_state.ensemble_variables,
            mc_state.config,
            mc_state.potential_variables,
            mc_state.dist2_mat,
            mc_state.new_dist2_vec,
            mc_state.en_tot,
            pot
        )
    else
        error("get_energy_neigh! currently only supports atommove")
    end

    return mc_state
end

function acc_test_with_rand!(mc_state::MCState, ensemble, movetype::String, r_accept::Float64)
    if metropolis_condition(movetype, mc_state, ensemble) >= r_accept
        swap_config!(mc_state, movetype)
    end

    return mc_state
end

function mc_move_neigh!(mc_state, move_strat, pot, ensemble)
    N = length(mc_state.config.pos)

    mc_state.ensemble_variables.index = rand(1:N)
    movetype = move_strat.movestrat[mc_state.ensemble_variables.index]

    mc_state = generate_move!(mc_state, movetype)
    mc_state = get_energy_neigh!(mc_state, pot, movetype)
    acc_test!(mc_state, ensemble, movetype)

    return mc_state
end

function mc_step_neigh_single!(
    mc_state,
    move_strat,
    pot,
    ensemble,
    n_steps::Int,
)
    for _ in 1:n_steps
        mc_state = mc_move_neigh!(
            mc_state,
            move_strat,
            pot,
            ensemble,
        )
    end

    return mc_state
end

function mc_step_old_single!(mc_state, move_strat, pot, ensemble, n_steps::Int)
    for i_step in 1:n_steps
        mc_state = mc_move!(mc_state, move_strat, pot, ensemble)
    end

    return mc_state
end

function paired_mc_step_compare!(mc_state_old, mc_state_neigh, move_strat, pot, ensemble, n_steps::Int)
    trial_energy_diffs = Float64[]
    trial_g_diffs = Float64[]
    accepted_flags = Bool[]

    for step in 1:n_steps
        N = length(mc_state_old.config.pos)

        # Same atom selected for both methods
        atomindex = rand(1:N)
        movetype = move_strat.movestrat[atomindex]

        mc_state_old.ensemble_variables.index = atomindex
        mc_state_neigh.ensemble_variables.index = atomindex

        # Generate trial move using old state
        mc_state_old = generate_move!(mc_state_old, movetype)

        # Copy the exact same proposed move into neighbour state
        mc_state_neigh.ensemble_variables.trial_move = mc_state_old.ensemble_variables.trial_move
        mc_state_neigh.new_dist2_vec = copy(mc_state_old.new_dist2_vec)

        # Compute trial energies
        mc_state_old = get_energy!(mc_state_old, pot, movetype)
        mc_state_neigh = get_energy_neigh!(mc_state_neigh, pot, movetype)

        push!(trial_energy_diffs, abs(mc_state_old.new_en - mc_state_neigh.new_en))
        push!(trial_g_diffs, maximum(abs.(
            mc_state_old.potential_variables.new_g_matrix .-
            mc_state_neigh.potential_variables.new_g_matrix
        )))

        # Same accept/reject random number
        r_accept = rand()

        old_energy_before = mc_state_old.en_tot

        acc_test_with_rand!(mc_state_old, ensemble, movetype, r_accept)
        acc_test_with_rand!(mc_state_neigh, ensemble, movetype, r_accept)

        push!(accepted_flags, mc_state_old.en_tot != old_energy_before)
    end

    return (
        mc_state_old = mc_state_old,
        mc_state_neigh = mc_state_neigh,
        max_trial_energy_diff = maximum(trial_energy_diffs),
        max_trial_g_diff = maximum(trial_g_diffs),
        final_energy_diff = abs(mc_state_old.en_tot - mc_state_neigh.en_tot),
        final_g_diff = maximum(abs.(
            mc_state_old.potential_variables.g_matrix .-
            mc_state_neigh.potential_variables.g_matrix
        )),
        n_accepted = count(accepted_flags)
    )
end

# ------------------------------------------------------------
# Build Cu cluster test state
# ------------------------------------------------------------

shell = 6
target_nn_bohr = 4.575916480942233

positions_raw = deduplicate_positions(mackay_icosahedron(shell))
raw_nn = central_neighbour_distance_scale(positions_raw)
positions = [(target_nn_bohr / raw_nn) * p for p in positions_raw]

N = length(positions)
cluster_radius = maximum(norm(p) for p in positions)
bc = SphericalBC(radius = cluster_radius + 5.0)
config = Config(positions, bc)

ensemble = NVT(N)
move_strat = MoveStrategy(ensemble)

temp = 1000.0
beta = 1.0 / temp

mc_state = MCState(temp, beta, config, ensemble, pot)

# ------------------------------------------------------------
# Timing test: old vs neighbour one-cycle runtime
# ------------------------------------------------------------

n_timing_repeats = 5

neigh_times = Float64[]

for r in 1:n_timing_repeats
    seed = 1234 + r

    Random.seed!(seed)
    t_neigh = @elapsed mc_step_neigh_single!(
        deepcopy(mc_state),
        move_strat,
        pot,
        ensemble,
        N
    )

    push!(neigh_times, t_neigh)
end

neigh_time = median(neigh_times)

println()
println("cu923 neighbour-rule runtimes for one MCstep")
println("attempted moves = ", N)
println("timing repeats = ", n_timing_repeats)
println("Cu923 one-cycle timing")
println("neigh_times = ", neigh_times)
println("neigh median time = ", neigh_time, " s")


cu923 neighbour-rule runtimes for one MCstep
attempted moves = 923
timing repeats = 5
Cu549 one-cycle timing
neigh_times = [12.243024292, 12.200698655, 9.980906095, 10.638501375, 9.862675922]
neigh median time = 10.638501375 s


In [5]:
using Pkg
Pkg.activate("/home/rev/ParallelTemperingMonteCarlo.jl")
Pkg.instantiate()
using ParallelTemperingMonteCarlo
using Random, DelimitedFiles
using Random
using Printf
using Statistics



function mc_step_neigh!(
    mc_states::MCStateVector,
    move_strat::MoveStrategy{N,E},
    pot::Ptype,
    ensemble::Etype,
    n_steps::Int,
) where {N,E,Ptype,Etype}

    Threads.@threads for state in mc_states
        mc_step_neigh_single!(
            state,
            move_strat,
            pot,
            ensemble,
            n_steps,
        )
    end

    return mc_states
end


function mc_cycle_neigh!(
    mc_states::MCStateVector,
    move_strat::MoveStrategy{N,E},
    mc_params::MCParams,
    pot::Ptype,
    ensemble::Etype,
    n_steps::Int,
    index::Int,
) where {N,E,Ptype,Etype}

    mc_states = mc_step_neigh!(
        mc_states,
        move_strat,
        pot,
        ensemble,
        n_steps,
    )

    if rand() < 0.1
        parallel_tempering_exchange!(
            mc_states,
            mc_params,
            ensemble,
        )
    end

    if rem(index, mc_params.n_adjust) == 0
        for state in mc_states
            update_max_stepsize!(
                state,
                mc_params.n_adjust,
                ensemble,
                mc_params.min_acc,
                mc_params.max_acc,
            )
        end
    end

    return mc_states
end


function mc_cycle_neigh!(
    mc_states::MCStateVector,
    move_strat::MoveStrategy{N,E},
    mc_params::MCParams,
    pot::Ptype,
    ensemble::Etype,
    n_steps::Int,
    results::Output,
    idx::Int,
    rdfsave::Bool,
) where {N,E,Ptype,Etype}

    mc_states = mc_cycle_neigh!(
        mc_states,
        move_strat,
        mc_params,
        pot,
        ensemble,
        n_steps,
        idx,
    )

    if rem(idx, mc_params.mc_sample) == 0
        sampling_step!(
            mc_params,
            mc_states,
            ensemble,
            idx,
            results,
            rdfsave,
        )
    end

    return mc_states
end


function equilibration_cycle_neigh!(
    mc_states::MCStateVector,
    move_strat::MoveStrategy{N,E},
    mc_params::MCParams,
    pot::Ptype,
    ensemble::Etype,
    n_steps::Int,
    results::Output,
) where {N,E,Ptype,Etype}

    ebounds = [100.0, -100.0]

    for i in 1:mc_params.eq_cycles
        mc_states = mc_cycle_neigh!(
            mc_states,
            move_strat,
            mc_params,
            pot,
            ensemble,
            n_steps,
            i,
        )

        for state in mc_states
            ebounds = check_e_bounds(
                state.en_tot,
                ebounds,
            )
        end
    end

    for state in mc_states
        reset_counters(state)
    end

    results = initialise_histograms!(
        mc_params,
        results,
        ebounds,
        mc_states[1].config.bc,
    )

    return mc_states, results
end


function equilibration_neigh!(
    mc_states::MCStateVector,
    move_strat::MoveStrategy{N,E},
    mc_params::MCParams,
    pot::Ptype,
    ensemble::Etype,
    n_steps::Int,
    results::Output,
    restart::Bool,
) where {N,E,Ptype,Etype}

    for state in mc_states
        push!(state.ham, 0)
        push!(state.ham, 0)
    end

    if restart
        return mc_states, results
    else
        return equilibration_cycle_neigh!(
            mc_states,
            move_strat,
            mc_params,
            pot,
            ensemble,
            n_steps,
            results,
        )
    end
end


function ptmc_run_neigh!(
    mc_params::MCParams,
    temp::TempGrid,
    start_config::Config,
    potential::Ptype,
    ensemble::Etype;
    rdfsave::Bool=false,
    restart::Bool=false,
    save=false,
    workingdirectory=pwd(),
) where {Ptype,Etype}

    cd(workingdirectory)

    if save != false
        save_init(
            potential,
            ensemble,
            mc_params,
            temp,
        )
    end

    mc_states,
    move_strategy,
    results,
    n_steps,
    start_counter = initialisation(
        mc_params,
        temp,
        start_config,
        potential,
        ensemble,
    )

    println("params set")

    mc_states, results = equilibration_neigh!(
        mc_states,
        move_strategy,
        mc_params,
        potential,
        ensemble,
        n_steps,
        results,
        restart,
    )

    if save != false
        save_histparams(results)
    end

    println("equilibration complete")

    for i in start_counter:mc_params.mc_cycles
        mc_states = mc_cycle_neigh!(
            mc_states,
            move_strategy,
            mc_params,
            potential,
            ensemble,
            n_steps,
            results,
            i,
            rdfsave,
        )

        if save == false
            # No checkpoint.
        elseif rem(i, save) == 0
            checkpoint(
                i,
                mc_states,
                results,
                ensemble,
                rdfsave,
            )
        end
    end

    println("MC loop done.")

    results = finalise_results(
        mc_states,
        mc_params,
        results,
    )

    println("done")

    return mc_states, results
end

function get_energy_neigh!(mc_state, pot, movetype::String)
    if movetype == "atommove"
        mc_state.potential_variables, mc_state.new_en = energy_update_neigh!(
            mc_state.ensemble_variables,
            mc_state.config,
            mc_state.potential_variables,
            mc_state.dist2_mat,
            mc_state.new_dist2_vec,
            mc_state.en_tot,
            pot
        )
    else
        error("get_energy_neigh! currently only supports atommove")
    end

    return mc_state
end

function acc_test_with_rand!(mc_state::MCState, ensemble, movetype::String, r_accept::Float64)
    if metropolis_condition(movetype, mc_state, ensemble) >= r_accept
        swap_config!(mc_state, movetype)
    end

    return mc_state
end

function mc_move_neigh!(mc_state, move_strat, pot, ensemble)
    N = length(mc_state.config.pos)

    mc_state.ensemble_variables.index = rand(1:N)
    movetype = move_strat.movestrat[mc_state.ensemble_variables.index]

    mc_state = generate_move!(mc_state, movetype)
    mc_state = get_energy_neigh!(mc_state, pot, movetype)
    acc_test!(mc_state, ensemble, movetype)

    return mc_state
end

function neighbour_union_from_cutoff_vectors(f_old_row, f_new_vec, atomindex)
    neighbours = Int[]

    for j in eachindex(f_new_vec)
        if j != atomindex && (f_old_row[j] != 0.0 || f_new_vec[j] != 0.0)
            push!(neighbours, j)
        end
    end

    return neighbours
end

  Activating project at `~/ParallelTemperingMonteCarlo.jl`


neighbour_union_from_cutoff_vectors (generic function with 1 method)

In [ ]:
using ParallelTemperingMonteCarlo
using Random, DelimitedFiles

#demonstration of the new version of the new code   
script_folder = @__DIR__ # folder where this script is located
data_path = joinpath(script_folder, "data") # path to data files, so "./data/"
#-------------------------------------------------------#
#-----------------------MC Params-----------------------#
#-------------------------------------------------------#


Random.seed!(1234)
n_atoms = 55
ti = 400.
tf = 1200.
n_traj = 28

temp = TempGrid{n_traj}(ti,tf) 

# MC simulation details

mc_cycles = 10  #default 20% equilibration cycles on top


mc_sample = 1  #sample every mc_sample MC cycles

#move_atom=AtomMove(n_atoms) #move strategy (here only atom moves, n_atoms per MC cycle)
displ_atom = 0.1 # Angstrom
n_adjust = 100

max_displ_atom = [0.1*sqrt(displ_atom*temp.t_grid[i]) for i in 1:n_traj]

mc_params = MCParams(mc_cycles, n_traj, n_atoms, mc_sample = mc_sample, n_adjust = n_adjust)


#-------------------------------------------------------------#
#----------------------Potential------------------------------#
#-------------------------------------------------------------#

evtohartree = 0.0367493
nmtobohr = 18.8973
#parameters taken from L Vocadlo etal J Chem Phys V120N6 2004
n = 8.482
m = 4.692
ϵ = evtohartree*0.0370
a = 0.25*nmtobohr
C = 27.561

pot = RuNNerPotential(nnp, radsymmvec, angularsymmvec)
#-------------------------------------------------------------#
#------------------RuNNer Potential---------------------------#
#-------------------------------------------------------------#
#-------------------------------------------#
#--------Vector of radial symm values-------#
#-------------------------------------------#
X = [ 11              0.001   0.000  11.338
 10              0.001   0.000  11.338
 11              0.020   0.000  11.338
 10              0.020   0.000  11.338
 11              0.035   0.000  11.338
 10              0.035   0.000  11.338
 11              0.100   0.000  11.338
 10              0.100   0.000  11.338
 11              0.400   0.000  11.338
 10              0.400   0.000  11.338]

radsymmvec = RadialType2{Float64}[]
angularsymmvec = AngularType3{Float64}[]

#--------------------------------------------#
#--------Vector of angular symm values-------#
#--------------------------------------------#
V = [[0.0001,1,1,11.338],[0.0001,-1,2,11.338],[0.003,-1,1,11.338],[0.003,-1,2,11.338],[0.008,-1,1,11.338],[0.008,-1,2,11.338],[0.008,1,2,11.338],[0.015,1,1,11.338],[0.015,-1,2,11.338],[0.015,-1,4,11.338],[0.015,-1,16,11.338],[0.025,-1,1,11.338],[0.025,1,1,11.338],[0.025,1,2,11.338],[0.025,-1,4,11.338],[0.025,-1,16,11.338],[0.025,1,16,11.338],[0.045,1,1,11.338],[0.045,-1,2,11.338],[0.045,-1,4,11.338],[0.045,1,4,11.338],[0.045,1,16,11.338],[0.08,1,1,11.338],[0.08,-1,2,11.338],[0.08,-1,4,11.338],[0.08,1,4,11.338]]

T = [111,110,100]

angularsymmvec = AngularType3{Float64}[]
#-------------------------------------------#
#-----------Including scaling data----------#
#-------------------------------------------#
file = open(joinpath(data_path,"scaling.data")) # full path "./data/scaling.data"
scalingvalues = readdlm(file)
close(file)
G_value_vec = []
for row in eachrow(scalingvalues[1:88,:])
    max_min = [row[4],row[3]]
    push!(G_value_vec,max_min)
end


for symmindex in eachindex(eachrow(X))
    row = X[symmindex,:]
    radsymm = RadialType2{Float64}(row[2],row[4],Int(row[1]),G_value_vec[symmindex])
    push!(radsymmvec,radsymm)
end


let n_index = 10

for element in V
    for types in T

        n_index += 1

        symmfunc = AngularType3{Float64}(element[1],element[2],element[3],11.338,types,G_value_vec[n_index])

        push!(angularsymmvec,symmfunc)
    end
end
end
#---------------------------------------------------#
#------concatenating radial and angular values------#
#---------------------------------------------------#

totalsymmvec = vcat(radsymmvec,angularsymmvec)


#--------------------------------------------------#
#-----------Initialising the nnp weights-----------#
#--------------------------------------------------#
num_nodes::Vector{Int32} = [88, 20, 20, 1]
activation_functions::Vector{Int32} = [1, 2, 2, 1]
file = open(joinpath(data_path, "weights.029.data"), "r+") # "./data/weights.029.data"
weights=readdlm(file)
close(file)
weights = vec(weights)
nnp = NeuralNetworkPotential(num_nodes,activation_functions,weights)

println(typeof(radsymmvec))
println(typeof(angularsymmvec))
println(eltype(radsymmvec))
println(eltype(angularsymmvec))

runnerpotential = RuNNerPotential(nnp,radsymmvec,angularsymmvec)




#-------------------------------------------------------------#
#------------------------Move Strategy------------------------#
#-------------------------------------------------------------#
ensemble = NVT(n_atoms)
move_strat = MoveStrategy(ensemble)
#-------------------------------------------------------------#
#-----------------------Starting Config-----------------------#
#-------------------------------------------------------------#

ico_55 = [[0.0000006584,       -0.0000019175,        0.0000000505],
[-0.0000005810,       -0.0000004871,        0.6678432175],
[0.1845874248,       -0.5681026047,        0.2986701538],
[-0.4832557457,       -0.3511072166,        0.2986684497],
[-0.4832557570,        0.3511046452,        0.2986669456],
[0.1845874064,        0.5681000550,        0.2986677202],
[0.5973371920,       -0.0000012681,        0.2986697030],
[-0.1845860897,       -0.5681038901,       -0.2986676192],
[-0.5973358752,       -0.0000025669,       -0.2986696020],
[-0.1845861081,        0.5680987696,       -0.2986700528],
[0.4832570624,        0.3511033815,       -0.2986683486],
[0.4832570738,       -0.3511084803,       -0.2986668445],
[0.0000018978,       -0.0000033480,       -0.6678431165],
[-0.0000017969,        0.0000009162,        1.3230014650],
[0.1871182835,       -0.5758942175,        0.9797717078],
[-0.4898861924,       -0.3559221410,       0.9797699802],
[-0.4898862039,        0.3559224872,        0.9797684555],
[0.1871182648,        0.5758945856,        0.9797692407],
[0.6055300485,        0.0000001908,        0.9797712507],
[0.7926501864,       -0.5758950093,        0.6055339635],
[0.3656681761,       -1.1254128670,        0.5916673591],
[-0.3027660545,       -0.9318173412,        0.6055326929],
[-0.9573332453,       -0.6955436707,        0.5916639831],
[-0.9797705418,       -0.0000006364,        0.6055294407],
[-0.9573332679,        0.6955423392,        0.5916610035],
[-0.3027660847,        0.9318160902,        0.6055287012],
[0.3656681396,        1.1254115783,        0.5916625380],
[0.7926501677,        0.5758937939,        0.6055314964],
[1.1833279992,       -0.0000006311,        0.5916664660],
[0.6770051458,       -0.9318186223,        0.0000033028],
[0.0000006771,       -1.1517907207,        0.0000025175],
[-0.6770037988,       -0.9318186442,        0.0000007900],
[-1.0954155825,       -0.3559242494,       -0.0000012200],
[-1.0954155940,        0.3559203788,       -0.0000027447],
[-0.6770038290,        0.9318147872,       -0.0000032017],
[0.0000006397,        1.1517868856,       -0.0000024165],
[0.6770051155,        0.9318148091,       -0.0000006889],
[1.0954168993,        0.3559204143,        0.0000013211],
[1.0954169108,       -0.3559242139,        0.0000028458],
[0.3027674014,       -0.9318199253,       -0.6055286002],
[-0.3656668229,       -1.1254154134,       -0.5916624370],
[-0.7926488510,       -0.5758976290,       -0.6055313954],
[-1.1833266824,       -0.0000032040,       -0.5916663649],
[-0.7926488697,        0.5758911742,       -0.6055338624],
[-0.3656668594,        1.1254090319,       -0.5916672580],
[0.3027673712,        0.9318135061,       -0.6055325919],
[0.9573345621,        0.6955398357,       -0.5916638820],
[0.9797718586,       -0.0000031986,       -0.6055293396],
[0.9573345846,       -0.6955461743,       -0.5916609025],
[-0.1871169480,       -0.5758984207,       -0.9797691397],
[-0.6055287318,       -0.0000040259,       -0.9797711497],
[-0.1871169667,        0.5758903824,       -0.9797716067],
[0.4898875091,        0.3559183059,       -0.9797698792],
[0.4898875207,       -0.3559263223,       -0.9797683545],
[0.0000031136,       -0.0000047513,       -1.3230013639]]
#convert to Bohr

copperconstant = 0.36258*nmtobohr
pos_cu55 = copperconstant*ico_55

AtoBohr = 1.8897259886

length(pos_cu55) == n_atoms || error("number of atoms and positions not the same - check starting config")


#histogram information
n_bin = 100

#boundary conditions
bc_cu55 = SphericalBC(radius=14*AtoBohr)   #5.32 Angstrom
start_config = Config(pos_cu55, bc_cu55)


#@profview ptmc_run!(mc_params,temp,start_config,pot,ensemble)

#rm("checkpoint",recursive=true)

println(typeof(runnerpotential))
println(typeof(nnp))

states,results = ptmc_run!(mc_params,temp,start_config,runnerpotential,ensemble;save=1000)

In [ ]:
#the below is the allocation-free and most efficient version of function neighbour_union_from_cutoff_vectors!
function neighbour_union_from_cutoff_vectors!(
    neighbours::Vector{Int},
    f_old_row,
    f_new_vec,
    atomindex::Int,
)
    k = 0

    @inbounds for j in eachindex(f_new_vec)
        if j != atomindex &&
           (!iszero(f_old_row[j]) || !iszero(f_new_vec[j]))

            k += 1
            neighbours[k] = j
        end
    end

    return k
end

Base.Meta.ParseError: ParseError:
# Error @ /home/rev/ParallelTemperingMonteCarlo.jl/ParallelTemperingMonteCarlo.jl/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y123sdnNjb2RlLXJlbW90ZQ==.jl:6:1
    atomindex::Int,
)
╙ ── unexpected `)`